# 축3. 자가소비 적합성 분석
**서울시 공영주차장 태양광 캐노피 우선입지 선정 연구**

---

## 목차

1. 데이터 로드
2. 실증근거 분석
   - 2.1 태양광-EV 충전 연계 현실화
   - 2.2 EV 충전소 변수 설계 근거
   - 2.3 EV 충전소 변수 반경 설정 근거
   - 2.4 노외 vs 노상 자체 전력소비 차이
3. 변수 설계
   - 3.1 EV 충전소 반경별 분포 분석 → ev_50m, ev_200m 확정
   - 3.2 최종 변수 구성
   - 3.3 ev_충전기_설치수(50m) 프록시 신뢰도 검증
     - 환경부 API 검증 시도 및 한계
     - 서울시설공단 데이터 기반 검증
4. 가중치 산정
   - 4.1 CRITIC 시도 및 한계 확인
     - 4.1.1 자체소비 그룹 — 이진변수 문제
     - 4.1.2 EV충전 그룹 — 집계단위 불일치 문제
   - 4.2 자체소비 그룹 — 상관관계 + 민감도 분석
   - 4.3 EV충전 그룹 — 상관관계 + 민감도 분석
   - 4.4 하위축 간 가중치 — CRITIC
   - 4.5 최종 가중치 요약
5. 축3 점수 산출
   - 5.1 점수화 공식
   - 5.2 점수 분포 요약
   - 5.3 결과 저장
6. 시각화 및 인사이트
   - 6.1 축3 점수 지도
   - 6.2 자체소비 vs EV충전 산점도 (4분면)
   - 6.3 자치구별 평균 축3 점수
   - 6.4 노상/노외별 점수 분포 박스플롯
   - 6.5 상위 20개 주차장 상세 비교
   - 6.6.1 보완 분석 — ev_충전기_설치수 프록시 정확도 검토1
   - 6.6.2 보완 분석 — ev_충전기_설치수 프록시 정확도 검토2
7. 분석 요약 및 결론
   - 7.1 분석 목적 및 범위 요약
   - 7.2 최종 가중치 확정
   - 7.3 점수 분포 요약
   - 7.4 핵심 발견
   - 7.5 분석 한계
   - 7.6 다음 단계



## 분석 개요

태양광 캐노피에서 생산된 전력은 한전 계통망으로 연결되어 재분배되는 구조를 가진다.
본 분석에서는 태양광 전력의 **직접 소비 가능한 두 가지 경로**로 분석 범위를 정의한다.

- **주차장 자체 소비**: 조명, CCTV, 출입기, 안내판 등 설비 전력
- **EV 충전기 직접 공급**: 태양광 → 인버터 → 배전반 → EV 충전기 연계

> **핵심 전제**: 이미 EV 충전기가 설치된 주차장에 태양광 캐노피를 설치하면,  
> 기존 충전 인프라와 즉시 연계되어 자가소비율을 높일 수 있다.  
> 근거: 서울시 2021년 공영주차장 592기 EV 충전기 설치 계획 (2022년 상반기 완료),  
> 서울에너지공사 솔라스테이션 태양광-EV 충전 직접 연계 운영 사례

### 분석 대상
- 서울시 공영주차장 **741개소** (축2 전처리 데이터를 축3에 맞춰 재전처리,필요한 칼럼만 추가)

### 사용 데이터
| 데이터 | 출처 | 활용 변수 |
|--------|------|----------|
| 공영주차장 전처리 데이터 | 축2 전처리-> 축3 전처리 결과물 | 주차장명, 위경도, 노상/노외 |
| EV 충전소 현황 | 한국환경공단 공공데이터 API | 충전소 좌표 (74,024개) |
| 자치구별 전기차 등록현황 | 서울시 OA-15640 (2026.3 기준) | 자치구별 EV 등록대수 |
| 공영주차장 안내정보 원본 | 서울시 OA-13122 | 운영시간 (HHMM 형식) |

---
## 1. 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 전처리 완료 데이터
df = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\parking_axis3_preprocessed.csv',
    encoding='utf-8-sig'
)

# EV 충전소 데이터
df_ev = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\서울_EV충전소.csv',
    encoding='utf-8-sig'
)

# 태양광 설치 주차장
df_solar = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\solar_parking_list.csv',
    encoding='utf-8-sig'
)

print(f"주차장 데이터: {df.shape}")
print(f"EV 충전소: {df_ev.shape}")
print(f"태양광 설치 주차장: {df_solar.shape}")
print(f"\n태양광 컬럼: {df_solar.columns.tolist()}")

주차장 데이터: (741, 10)
EV 충전소: (74024, 37)
태양광 설치 주차장: (74, 4)

태양광 컬럼: ['개소명', '주소', '설치년도', '설치용량_KW']


---
## 2. 실증근거 분석

변수 설계에 앞서 네 가지 실증근거를 확보하여 분석의 논리적 타당성을 보강한다.

### 2.2 실증근거  태양광-EV 충전 연계 현실화

**목적**: EV충전 하위축 설계 정당성

**근거**:

**1. 서울에너지공사 솔라스테이션**
- 태양광 + ESS + EV 충전기를 결합한 복합 충전 인프라로, 태양광 전력을 직접 전기차 충전에 사용하거나 ESS에 저장 후 활용하는 구조로 운영 중 -> 뿐만 아니라 가정에서도 태양광 패널로 전기 자동차 충전 가능함 
- 출처: [text](https://www.i-se.co.kr/renew05) [text](https://joca-cable.com/ko/blog/solar-power-ev-charger/)

**2. 2026 재생에너지 금융지원사업**
- 태양광-EV 연계 프로젝트 중소기업 지원비율 80% 이내(기존 75%), 정책연계형(공영주차장 등) 5%p 상향, 풍력 시설자금 한도 800억원으로 확대 지원
- 출처: [text](https://www.knrec.or.kr/biz/pds/businoti/view.do?no=6364)

**결과 요약**:
- 실제 운영 사례 존재

    - 서울에너지공사의 ‘솔라스테이션’은 태양광 + ESS + EV 충전기를 결합한 복합 충전 인프라로,
    태양광으로 생산된 전력을 직접 전기차 충전에 사용하거나 ESS에 저장 후 활용하는 구조로 운영되고 있음

- 정책 확대 지원 진행 중

**핵심 결론**:
- 태양광 전력을 EV 충전에 직접 활용하는 구조는 이미 실증·운영되고 있으며, 정책적으로도 확대 지원 중인 현실적인 인프라 모델이다.

### 2.2 실증근거  EV 충전소 변수 설계 근거

**목적**: 주차장 내부 EV 충전기 수를 직접 확인할 수 있는 공개 데이터가 없어
거리 기반 프록시 변수 두 가지를 설계한 근거 마련

**근거**:

**변수 1 — ev_충전기_설치수 (50m, 엄격 연계)**
- 태양광 전력의 직접 공급은 '같은 전기사용계약·전기안전 범위' 내에서만
  현실적으로 가능 (리서치 근거: 서울시 공영주차장 기술 가이드라인)
- 주차장 내부 EV 충전기 설치 현황 데이터가 미공개이므로 좌표 기반 50m 이내
  충전소 수를 내부 충전기의 대리변수로 사용
- **이름 기반 필터링이 아닌 좌표 기반을 채택한 이유**:
  환경부 API 충전소명은 주차장명과 상이하거나(예: '수서역 공영주차장 복합충전소'),
  주차장 외 시설명으로 등록된 경우가 혼재함.
  이름 매칭 시 누락·오매칭 위험이 크고 정규화 기준이 자의적이 될 수 있어,
  전체 741개에 동일 기준을 일관되게 적용할 수 있는 좌표 기반 방식을 채택

**변수 2 — ev_200m (200m, EV 수요 밀도)**
- 직접 전력 공급 변수가 아닌 **EV 충전 수요 밀도** 변수로 프레이밍
- 반경 200m 내 충전소가 많다 → 해당 지역 EV 이용자가 많다
  → 태양광 EV 충전 수요가 높은 입지
- 200m 선정 근거:
  - GIS 공간분석 관행상 100~300m 버퍼를 인접 시설 정의에 활용
  - 도보 2~3분 거리로 생활권 충전 수요 반영에 적합
  - 100m: 0개 비율 57%로 변별력 낮음 / 300m: 0개 3.6%로 과포화
  - 200m에서 0개 15%, std 24.9로 변별력 최적
  (출처: 한국품질경영학회 논문, kpaj.or.kr 도보 접근성 논문)

### 2.3 실증근거  EV 충전소 변수 반경 설정 근거

**목적**: ev_충전기_설치수 (50m, 엄격 연계) 및 ev_200m (200m, EV 수요 밀도) EV 충전소 집계 시 사용한 반경 기준의 정당성 확보

**근거**:
- 좌표 기반 데이터의 구조적 한계 반영
    - EV 충전기와 공영주차장은 서로 다른 기준으로 좌표가 구축되어
    동일 위치라도 수십 미터(약 10~40m) 오차 발생
    따라서 0m 기준은 현실적으로 동일 위치 판단이 불가능
- 주차장 공간 규모 고려
    - 공영주차장은 최소 30~50m, 대형은 100m 이상 규모로
    내부 시설 간에도 좌표 거리 발생
    - 즉, 일정 반경 없이 내부 여부를 판단하면 과소추정 위험 존재

- 50m 선정의 타당성
    1. 좌표 오차 + 주차장 실제 면적을 동시에 반영한 값으로
        -  ‘실질적 동일 부지’ 판단을 위한 현실적 기준
    2. 민감도 분석 결과: 50m가 가장 효과적
    3. 서울시 보도자료(2021)에 따르면 주차장당 평균 약 1기 수준에 불과함
        - 50m 집계 결과 평균 0.87개는 이러한 낮은 보급 현실과 부합함
- 200m 선정의 타당성
    1. GIS 공간분석 관행상 인접 시설 정의 시 100~300m 버퍼가 일반적
        - [text](https://www.jkst.or.kr/articles/xml/1L1W/)
        - GIS 공간분석 관행상 인접 시설 정의 시 100~300m 버퍼가 일반적이며, 이는 공영주차장 EV 충전소 입지 산정에도 적용 가능하다(한국교통학회, 2025).
jkst
    2. 또한 서울시 '생활권 5분 충전망' 도시 정책에서 EV 충전 접근성은
    300~500m 생활권 기준으로 설정됨
    3. 민감도 분석 결과: 200m가 가장 효과적
        -  도보 2~3분 수준의 실질적 이용 가능 거리


**핵심 결론**:
- 50m, 200m 기준은 데이터 오차, 공간 규모, GIS 관행, 정책 기준을 모두 반영한 합리적 거리 설정이다.

### 2.4 실증근거 노외 vs 노상 자체 전력소비 차이

**목적**: 노외 여부 가중치 부여의 정당성 근거 마련

**근거**: 주차장법에 따라 지하식·건축물식 노외주차장은 **상시 조명 유지 의무**가 있음  
(주차구획 최소 10럭스, 출입구 최소 300럭스 이상)

**결과요약**:
노외 주차장은 조명·환기·정산기 등 설비 전력을 상시 소비하므로  
운영시간이 동일하더라도 노상보다 자체 전력소비가 유의미하게 높다.

---
## 3. 변수 설계



### 3.1 EV 충전소 반경별 분포 분석

In [3]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # 지구 반지름 (m)

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [4]:
import pandas as pd
import numpy as np

# EV 충전소 재로드
df_ev = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\서울_EV충전소.csv',
    encoding='utf-8-sig'
)
df_ev['lat'] = pd.to_numeric(df_ev['lat'], errors='coerce')
df_ev['lng'] = pd.to_numeric(df_ev['lng'], errors='coerce')
df_ev = df_ev.dropna(subset=['lat', 'lng'])

ev_lat = df_ev['lat'].values
ev_lng = df_ev['lng'].values

print(f"EV 충전소: {len(df_ev)}개")
print(f"lat 범위: {ev_lat.min():.4f} ~ {ev_lat.max():.4f}")
print(f"lng 범위: {ev_lng.min():.4f} ~ {ev_lng.max():.4f}")

# 수서역북 테스트
test_row = df[df['pklt_nm'] == '수서역북 공영주차장'].iloc[0]
count = sum(
    haversine(test_row['lat'], test_row['lot'], lat, lng) <= 30
    for lat, lng in zip(ev_lat, ev_lng)
)
print(f"\n수서역북 30m 이내: {count}개")

EV 충전소: 74024개
lat 범위: 33.0000 ~ 37.9165
lng 범위: 125.0000 ~ 129.3607

수서역북 30m 이내: 4개


**민감도분석으로 거리정하기**

In [5]:
import numpy as np

def haversine_vec(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1[:, None]
    dlon = lon2 - lon1[:, None]
    
    a = np.sin(dlat/2)**2 + np.cos(lat1[:, None]) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# 거리 행렬 (주차장 x 충전소)
dist_matrix = haversine_vec(
    df['lat'].values,
    df['lot'].values,   
    np.array(ev_lat),
    np.array(ev_lng)
)

In [6]:
radii = [30, 50, 70, 100, 200, 300, 400, 500]

for r in radii:
    counts = (dist_matrix <= r).sum(axis=1)
    
    df[f'ev_{r}m'] = counts
    has = (counts > 0).sum()
    mean = counts.mean()
    
    print(f"{r}m — 충전소 있는 주차장: {has}개 ({has/742*100:.1f}%), 평균: {mean:.2f}개")

30m — 충전소 있는 주차장: 58개 (7.8%), 평균: 0.31개
50m — 충전소 있는 주차장: 116개 (15.6%), 평균: 0.87개
70m — 충전소 있는 주차장: 164개 (22.1%), 평균: 1.50개
100m — 충전소 있는 주차장: 318개 (42.9%), 평균: 4.63개
200m — 충전소 있는 주차장: 630개 (84.9%), 평균: 21.19개
300m — 충전소 있는 주차장: 714개 (96.2%), 평균: 48.21개
400m — 충전소 있는 주차장: 734개 (98.9%), 평균: 88.62개
500m — 충전소 있는 주차장: 734개 (98.9%), 평균: 135.95개


#### 반경별 집계 결과

| 반경 | 충전소 있는 주차장 | 비율 | 평균 충전소 수 | 선정 |
|------|-----------------|------|--------------|------|
| 30m | 58개 | 7.8% | 0.31개 | |
| **50m** | **116개** | **15.6%** | **0.87개** | **✓ 내부 기준** |
| 70m | 164개 | 22.1% | 1.49개 | |
| 100m | 318개 | 42.9% | 4.6개 | |
| **200m** | **631개** | **85.0%** | **21.2개** | **✓ 근접 기준** |
| 300m | 715개 | 96.4% | 48.2개 | |
| 400m | 735개 | 99.1% | 88.5개 | |


#### 분포확인

In [7]:
# 분포확인
print("=== 30m 분포 ===")
print(df['ev_30m'].describe())
print(f"\n0개: {(df['ev_30m']==0).sum()}개")
print(f"10개 미만: {(df['ev_30m']<10).sum()}개")
print(f"50개 이상: {(df['ev_30m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_30m']>=100).sum()}개")

print("=== 50m 분포 ===")
print(df['ev_50m'].describe())
print(f"\n0개: {(df['ev_50m']==0).sum()}개")
print(f"10개 미만: {(df['ev_50m']<10).sum()}개")
print(f"50개 이상: {(df['ev_50m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_50m']>=100).sum()}개")


print("=== 70m 분포 ===")
print(df['ev_70m'].describe())
print(f"\n0개: {(df['ev_70m']==0).sum()}개")
print(f"10개 미만: {(df['ev_70m']<10).sum()}개")
print(f"50개 이상: {(df['ev_70m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_70m']>=100).sum()}개")


print("\n=== 100m 분포 ===")
print(df['ev_100m'].describe())
print(f"\n0개: {(df['ev_100m']==0).sum()}개")
print(f"10개 미만: {(df['ev_100m']<10).sum()}개")
print(f"50개 이상: {(df['ev_100m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_100m']>=100).sum()}개")

print("\n=== 200m 분포 ===")
print(df['ev_200m'].describe())
print(f"\n0개: {(df['ev_200m']==0).sum()}개")
print(f"10개 미만: {(df['ev_200m']<10).sum()}개")
print(f"50개 이상: {(df['ev_200m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_200m']>=100).sum()}개")

print("\n=== 300m 분포 ===")
print(df['ev_300m'].describe())
print(f"\n0개: {(df['ev_300m']==0).sum()}개")
print(f"10개 미만: {(df['ev_300m']<10).sum()}개")
print(f"50개 이상: {(df['ev_300m']>=50).sum()}개")
print(f"100개 이상: {(df['ev_300m']>=100).sum()}개")

=== 30m 분포 ===
count    741.000000
mean       0.313090
std        1.623877
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       22.000000
Name: ev_30m, dtype: float64

0개: 683개
10개 미만: 734개
50개 이상: 0개
100개 이상: 0개
=== 50m 분포 ===
count    741.000000
mean       0.869096
std        3.543228
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       62.000000
Name: ev_50m, dtype: float64

0개: 625개
10개 미만: 720개
50개 이상: 1개
100개 이상: 0개
=== 70m 분포 ===
count    741.000000
mean       1.495277
std        4.595714
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       62.000000
Name: ev_70m, dtype: float64

0개: 577개
10개 미만: 702개
50개 이상: 1개
100개 이상: 0개

=== 100m 분포 ===
count    741.000000
mean       4.632928
std       11.210044
min        0.000000
25%        0.000000
50%        0.000000
75%        4.000000
max      150.000000
Name: ev_100m, dtype: float64

0개: 423개
10개 미만: 633개
50개 이상: 4개
100개 이상: 

### 반경 기준 확정

**내부 변수: 50m 확정**
- 30m: 0개 비율 92.2% — 변별력 없음
- 70m~100m: 주차장 외부 충전소 혼입 가능성 증가
- 50m: 좌표 오차(10~40m) + 주차장 면적(30~50m)을 동시에 고려한
  현실적 내부 판별 기준. 0개 비율 84.4%는 실제 공영주차장 내 EV 충전기가 없는 주차장이 많다는 현실을 반영한 결과

**근접 변수 (EV 수요 밀도): 200m 확정**
- 100m: 100m는 0개 비율이 57.1%로 여전히 sparse하고, EV 수요 밀도 변수로서 주차장 간 충전 인프라 집적도 차이를 포착하기엔 변별력이 부족함
- 300m 이상: 0개 3.6% 이하로 과포화 — 수요 밀도 차별화 불가
- 200m: 0개 15%, std 24.9로 주차장 간 EV 수요 밀도 차이를
  가장 효과적으로 포착. 도보 2~3분 거리로 생활권 충전 수요 반영에 적합

#### 확정
- **내부 기준 50m**: 좌표 오차(10~40m) + 주차장 면적 고려한 현실적 내부 기준
- **근접 기준 200m**: 도보 2~3분 이내, 0개 비율 15%로 변별력 최적 (300m는 0개 3.6%로 변별력 낮음)

### 3.2 최종 변수 구성

| 하위축 | 변수명 | 설명 | 방향 |
|--------|--------|------|------|
| 자체소비 | 운영시간 (가중평균) | (평일×5 + 주말×2) / 7 | ↑ |
| 자체소비 | 노외여부 (0/1) | 노외=1, 노상=0 | ↑ |
| EV충전 | ev_충전기_설치수 (50m) | 주차장 내부 EV 충전기 수 | ↑ |
| EV충전 | ev_200m | 반경 200m 내 EV 충전소 수 | ↑ |
| EV충전 | ev_등록대수 | 자치구별 전기차 등록대수 | ↑ |

### 3.3 ev_충전기_설치수 (50m) 프록시 신뢰도 검증

#### 검증 데이터 선정 과정

처음에는 환경부 API(서울_EV충전소.csv)의 충전소명(statNm) 기반으로
실측 충전기 수를 산출하여 검증을 시도하였다.

- **시도 방법**: statNm별 chgerId nunique 합산으로 충전기 수 추정
- **결과**: 상관계수 0.559, 절대 평균 차이 4.50 (18개 매칭)

그러나 다음 두 가지 구조적 한계로 부적합하다고 판단하였다.

- **한계 1 — statNm 설계 불일치**: 환경부 API statNm은 주차장명이 아닌
  충전기 설치 장소명으로 등록되어 동일 충전소가 여러 주차장명에 중복 매칭되거나
  주차장과 전혀 다른 이름으로 등록된 경우 매칭 불가
- **한계 2 — 충전기 경계 불일치**: statNm 단위가 주차장 경계와 일치하지 않아
  주차장 외부 인근 충전기까지 합산되는 과대추정이 구조적으로 발생
  (천호역: 실측 8개 → 환경부 27개, 마포유수지: 실측 5개 → 환경부 20개 등)

**→ 주차장명 ↔ 충전기 수가 직접 매핑된 서울시설공단
공영주차장 전기차 충전정보(2025.12)를 최종 검증 데이터로 채택한다.**

---

#### 3.3.1 서울시설공단 공영주차장 전기차 충전정보 기반 검증

| 항목 | 내용 |
|------|------|
| 출처 | 서울시설공단_공영주차장_전기차_충전정보_20251202.csv |
| 내용 | 서울시설공단 직영 공영주차장의 전기차 충전 이용 이력 (2022.04 ~ 2025.12) |
| 원본 범위 | 서울시설공단 직영 53개소 중 ([text](https://www.sisul.or.kr/open_content/parking/guidance/info.jsp)) **28개소** (EV 충전기 보유·이용 기록 있는 주차장만 포함) |
| 최종 매칭 | 정규화 + 중복 제거 후 우리 데이터(741개)와 **25개 매칭** |
| 미매칭 사유 | 수서역남(폐쇄) → 분석 대상 제외 / 마포유수지·구로디지털단지역 → 741개 내 미포함 |
| 중복 처리 | 도봉산·영등포구청·용산주차빌딩 각 2개씩 매칭 → 이름 유사도 기준 1개씩 유지 |

**검증 목적**: 실제 EV 충전기가 설치된 주차장에서
50m 거리 기반 집계값(ev_충전기_설치수)이 실측 충전기 수를
얼마나 정확하게 추정하는지 정량적으로 평가한다.

In [8]:
import re

df_charge = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\raw\서울시설공단_공영주차장 전기차 충전정보_20251202.csv',
    encoding='cp949'
)

# ── 실측 충전기 수 산출 ───────────────────────────────────────────────────
# 주차장+충전소별 고유 충전기 번호 수 합산
ev_real = df_charge.groupby(['주차장명', '충전소 식별자'])['충전기 번호'].nunique().reset_index()
ev_real = ev_real.groupby('주차장명')['충전기 번호'].sum().reset_index()
ev_real.columns = ['pklt_nm_raw', 'ev_실측']

print(f"서울시설공단 데이터 내 주차장 수: {ev_real['pklt_nm_raw'].nunique()}개")
print(ev_real.sort_values('ev_실측', ascending=False).to_string())

서울시설공단 데이터 내 주차장 수: 28개
   pklt_nm_raw  ev_실측
20         잠실역     12
13        수서역북     11
25        학여울역     11
21          종묘     10
10        사당노외      8
24         천호역      8
12    수서역남(폐쇄)      7
19      용산주차빌딩      6
5         도봉산역      6
16       신천유수지      5
7        마포유수지      5
23         천왕역      5
6          동대문      4
15        신방화역      4
11         세종로      4
14        신대방역      3
26        한강진역      3
1          개화역      3
0         개화산역      3
22      창동역(서)      3
8          볕우물      2
9          복정역      2
4         당산노외      2
3         구파발역      2
2     구로디지털단지역      2
27       훈련원공원      2
18      영등포구청역      1
17          영남      1


In [9]:
# ── 정규화 함수 ───────────────────────────────────────────────────────────
# 주의: 공백 제거 후 역$ 제거 순서 필수
# (공영주차장 제거 후 공백이 남으면 역$ 패턴이 미적용됨)
def normalize_name_strict(name):
    name = str(name)
    name = re.sub(r'\(시\)|\(구\)|舊', '', name)
    name = re.sub(r'\([^)]*\)', '', name)
    name = re.sub(r'공영주차장|주차장|공영|노외|복합충전소|충전소', '', name)
    name = re.sub(r'\s+', '', name)  # 공백 먼저 제거
    name = re.sub(r'역$', '', name)  # 그 다음 역$ 제거
    return name.strip()

ev_real['nm_key'] = ev_real['pklt_nm_raw'].apply(normalize_name_strict)
df['nm_key2'] = df['pklt_nm'].apply(normalize_name_strict)

In [10]:
# ── 매칭 ─────────────────────────────────────────────────────────────────
ev_real_dedup = ev_real.drop_duplicates(subset='nm_key')
df_valid = df.merge(
    ev_real_dedup[['nm_key', 'ev_실측']],
    left_on='nm_key2',
    right_on='nm_key',
    how='inner'
)

# 중복 매칭 제거 (이름 유사도 기준 — 더 정확한 이름 유지)
drop_idx = df_valid[df_valid['pklt_nm'].isin([
    '도봉산 공영주차장',   # 도봉산역 공영주차장 유지
    '영등포구청',          # 영등포구청역 공영주차장 유지
    '용산주차빌딩'         # 용산주차빌딩 공영주차장 유지
])].index
df_valid_clean = df_valid.drop(index=drop_idx).reset_index(drop=True)

print(f"최종 매칭 주차장: {len(df_valid_clean)}개")
print(df_valid_clean[['pklt_nm', 'ev_50m', 'ev_실측']].to_string())

최종 매칭 주차장: 25개
         pklt_nm  ev_50m  ev_실측
0     개화산역 공영주차장       8      3
1      개화역 공영주차장      10      3
2     구파발역 공영주차장       6      2
3      당산노외공영주차장       2      2
4     도봉산역 공영주차장       9      6
5      동대문 공영주차장      22      4
6      볕우물 공영주차장       2      2
7      복정역 공영주차장       3      2
8      사당노외공영주차장       0      8
9      세종로 공영주차장      14      4
10    수서역북 공영주차장       8     11
11    신대방역 공영주차장       0      3
12    신방화역 공영주차장      16      4
13   신천유수지 공영주차장       0      5
14      영남 공영주차장      12      1
15  영등포구청역 공영주차장       6      1
16  용산주차빌딩 공영주차장      14      6
17     잠실역 공영주차장       0     12
18   종묘주차장 공영주차장       0     10
19  창동역(서) 공영주차장       3      3
20     천왕역 공영주차장      11      5
21     천호역 공영주차장      21      8
22     학여울 공영주차장       2     11
23    한강진역 공영주차장       5      3
24   훈련원공원 공영주차장      17      2


In [11]:
# ── 비교 통계 ─────────────────────────────────────────────────────────────
diff_clean = df_valid_clean['ev_50m'] - df_valid_clean['ev_실측']
df_valid_clean['차이'] = diff_clean
df_valid_clean['판정'] = diff_clean.apply(
    lambda x: '일치(±2)' if abs(x) <= 2 else ('과대추정' if x > 2 else '과소추정')
)

print(f"=== 50m 프록시 신뢰도 검증 최종 결과 ===")
print(f"매칭 주차장    : {len(df_valid_clean)}개")
print(f"평균 차이      : {diff_clean.mean():.2f}")
print(f"절대 평균 차이 : {diff_clean.abs().mean():.2f}")
print(f"상관계수       : {df_valid_clean['ev_50m'].corr(df_valid_clean['ev_실측']):.3f}")
print(f"\n일치(±2) : {(df_valid_clean['판정']=='일치(±2)').sum()}개 ({(df_valid_clean['판정']=='일치(±2)').mean()*100:.1f}%)")
print(f"과대추정  : {(df_valid_clean['판정']=='과대추정').sum()}개 ({(df_valid_clean['판정']=='과대추정').mean()*100:.1f}%)")
print(f"과소추정  : {(df_valid_clean['판정']=='과소추정').sum()}개 ({(df_valid_clean['판정']=='과소추정').mean()*100:.1f}%)")
print(f"\n판정별 상세:")
print(df_valid_clean[['pklt_nm', 'ev_50m', 'ev_실측', '차이', '판정']].to_string())

=== 50m 프록시 신뢰도 검증 최종 결과 ===
매칭 주차장    : 25개
평균 차이      : 2.80
절대 평균 차이 : 6.80
상관계수       : -0.165

일치(±2) : 5개 (20.0%)
과대추정  : 13개 (52.0%)
과소추정  : 7개 (28.0%)

판정별 상세:
         pklt_nm  ev_50m  ev_실측  차이      판정
0     개화산역 공영주차장       8      3   5    과대추정
1      개화역 공영주차장      10      3   7    과대추정
2     구파발역 공영주차장       6      2   4    과대추정
3      당산노외공영주차장       2      2   0  일치(±2)
4     도봉산역 공영주차장       9      6   3    과대추정
5      동대문 공영주차장      22      4  18    과대추정
6      볕우물 공영주차장       2      2   0  일치(±2)
7      복정역 공영주차장       3      2   1  일치(±2)
8      사당노외공영주차장       0      8  -8    과소추정
9      세종로 공영주차장      14      4  10    과대추정
10    수서역북 공영주차장       8     11  -3    과소추정
11    신대방역 공영주차장       0      3  -3    과소추정
12    신방화역 공영주차장      16      4  12    과대추정
13   신천유수지 공영주차장       0      5  -5    과소추정
14      영남 공영주차장      12      1  11    과대추정
15  영등포구청역 공영주차장       6      1   5    과대추정
16  용산주차빌딩 공영주차장      14      6   8    과대추정
17     잠실역 공영주차장       0     12 -12    과

#### 검증 결과 요약

| 지표 | 값 |
|------|---|
| 매칭된 주차장 수 | 25개 (741개의 3.4%) |
| 평균 차이 (50m - 실측) | +2.80 (과대추정 방향) |
| 절대 평균 차이 | 6.80개 |
| 상관계수 | -0.165 |
| 일치(±2) | 5개 (20.0%) |
| 과대추정 | 13개 (52.0%) |
| 과소추정 | 7개 (28.0%) |

**해석:**

상관계수 -0.165, 일치율 20%로 50m 프록시가 실제 충전기 수를
정확하게 추정하지 못하는 것으로 나타났다.

- **과대추정(52%) 원인**: 주차장 경계 밖 인근 건물·도로변 충전소 혼입
  역세권 주차장(동대문 +18, 훈련원공원 +15, 세종로 +10 등)에서 두드러짐
- **과소추정(28%) 원인**: 주차장 좌표와 충전소 좌표 간 등록 기준 불일치
  대형 주차장(잠실역 -12, 종묘 -10, 학여울 -9 등)에서 두드러짐
- **두 오류가 반대 방향으로 작용**: 반경을 줄이면 과소추정 심화,
  늘리면 과대추정 심화 → 반경 조정으로는 해결이 어려운 구조적 한계

**검증 범위의 한계:**
- 서울시설공단 직영 53개소 중 EV 충전기 보유 주차장(28개)만 검증 가능
- 최종 25개(741개의 3.4%) → 전체 대표성 제한적
- 단, 실제 충전기 보유 주차장에서 프록시 정확도를 직접 검증한
  유일한 데이터소스로 의미 있음

**→ 한계로 명시하되, 주차장별 실측 데이터 미공개로 50m 거리 기반 유지**

---
## 4. 가중치 산정 (CRITIC)

가중치는 임의로 설정하지 않고 **CRITIC(CRiteria Importance Through Intercriteria Correlation)** 방법을 통해 데이터 기반으로 도출하였다.

### CRITIC 방법 개요
CRITIC은 각 변수의 **변별력(표준편차)**과 **독립성(상관관계)**을 동시에 고려하여 가중치를 산출한다.

$$C_j = \sigma_j \sum_{k=1}^{n}(1 - r_{jk})$$

- $\sigma_j$: 변수 j의 표준편차 → 변별력이 클수록 중요도 ↑  
- $r_{jk}$: 변수 j와 k의 상관계수 → 다른 변수와 독립적일수록 중요도 ↑  
- 최종 가중치 = $C_j / \sum C_j$

CRITIC은 MinMax 정규화 후 표준편차와 상관계수만을 기반으로 가중치를 산출하며,
**변수의 측정 유형(이진/연속/집계단위)을 구분하지 않는다.**

### 산정 구조
| 단계 | 대상 | 방법 |
|------|------|------|
| Step 1 | 자체소비 그룹 내부 (운영시간, 노외여부) | CRITIC |
| Step 2 | EV충전 그룹 내부 (ev_충전기_설치수, ev_200m, ev_등록대수) | CRITIC |
| Step 3 | 하위축 간 (자체소비 점수 vs EV충전 점수) | CRITIC |

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# 노외여부 수치화
df['노외여부'] = df['pklt_knd_nm'].apply(lambda x: 1 if '노외' in str(x) else 0)

# ev_50m → ev_충전기_설치수로 rename
df['ev_충전기_설치수'] = df['ev_50m']

# CRITIC 함수 정의
def critic_weights(data: pd.DataFrame) -> pd.Series:
    """
    CRITIC 가중치 산출 함수
    - data: 정규화된 변수 DataFrame
    - return: 변수별 가중치 Series
    """
    # MinMax 정규화
    norm = pd.DataFrame(
        MinMaxScaler().fit_transform(data),
        columns=data.columns
    )
    # 표준편차 (변별력)
    std = norm.std()
    # 상관행렬
    corr = norm.corr()
    # 독립성 합산: sum(1 - r_jk)
    conflict = (1 - corr).sum()
    # CRITIC 지수
    C = std * conflict
    # 가중치
    weights = C / C.sum()
    return weights, C, std, corr

### 4.1 Step 1 — 자체소비 그룹 CRITIC

In [13]:
# 자체소비 변수 선택
cols_self = ['운영시간', '노외여부']
data_self = df[cols_self].copy()

weights_self, C_self, std_self, corr_self = critic_weights(data_self)

# 결과 출력
print("=== 자체소비 그룹 CRITIC 결과 ===\n")
print("[ 상관행렬 ]")
print(corr_self.round(3))

print("\n[ 변수별 지표 ]")
summary_self = pd.DataFrame({
    '표준편차': std_self.round(4),
    '독립성합산(1-r)': C_self.div(std_self).round(4),
    'CRITIC 지수(C)': C_self.round(4),
    '가중치': weights_self.round(4)
})
print(summary_self.to_string())

print(f"\n[ 최종 가중치 ]")
for col, w in weights_self.items():
    print(f"  {col}: {w:.4f} ({w*100:.1f}%)")

=== 자체소비 그룹 CRITIC 결과 ===

[ 상관행렬 ]
       운영시간   노외여부
운영시간  1.000  0.676
노외여부  0.676  1.000

[ 변수별 지표 ]
        표준편차  독립성합산(1-r)  CRITIC 지수(C)     가중치
운영시간  0.3940      0.3242        0.1277  0.4666
노외여부  0.4503      0.3242        0.1460  0.5334

[ 최종 가중치 ]
  운영시간: 0.4666 (46.7%)
  노외여부: 0.5334 (53.3%)


#### 자체소비 그룹 CRITIC 한계

CRITIC 결과 **노외여부(53.3%) > 운영시간(46.7%)** 으로 산출되었으나, 이는 실질적 중요도를 반영하지 못한다.

**한계 원인: 이진변수의 표준편차 왜곡**

| 구분 | 운영시간 | 노외여부 |
|------|---------|---------|
| 변수 유형 | 연속형 (7.1 ~ 24.0) | 이진형 (0 또는 1) |
| 정규화 후 표준편차 | 0.394 | 0.450 |
| 독립성 합산 | 0.324 | 0.324 |

- 두 변수의 상관계수가 0.676으로 높아 **독립성 합산값이 동일**
- 결국 표준편차(변별력)만으로 가중치가 결정됨
- 노외여부(이진)는 MinMax 정규화 시 0과 1 사이를 양극단으로 사용하여
  **연속형 변수보다 인위적으로 표준편차가 크게 산출**되는 구조적 문제 발생
- CRITIC은 변수 유형을 구분하지 않으므로 이진변수가 포함될 경우
  변별력이 과대평가될 수 있음

→ **이진변수가 포함된 자체소비 그룹은 CRITIC 적용 불가**

### 4.2 Step 2 — EV충전 그룹 CRITIC

In [14]:
# EV충전 변수 선택
cols_ev = ['ev_충전기_설치수', 'ev_200m', 'ev_등록대수']
data_ev = df[cols_ev].copy()

weights_ev, C_ev, std_ev, corr_ev = critic_weights(data_ev)

print("=== EV충전 그룹 CRITIC 결과 ===\n")
print("[ 상관행렬 ]")
print(corr_ev.round(3))

print("\n[ 변수별 지표 ]")
summary_ev = pd.DataFrame({
    '표준편차': std_ev.round(4),
    '독립성합산(1-r)': C_ev.div(std_ev).round(4),
    'CRITIC 지수(C)': C_ev.round(4),
    '가중치': weights_ev.round(4)
})
print(summary_ev.to_string())

print(f"\n[ 최종 가중치 ]")
for col, w in weights_ev.items():
    print(f"  {col}: {w:.4f} ({w*100:.1f}%)")

=== EV충전 그룹 CRITIC 결과 ===

[ 상관행렬 ]
            ev_충전기_설치수  ev_200m  ev_등록대수
ev_충전기_설치수       1.000    0.197   -0.005
ev_200m          0.197    1.000    0.038
ev_등록대수         -0.005    0.038    1.000

[ 변수별 지표 ]
              표준편차  독립성합산(1-r)  CRITIC 지수(C)     가중치
ev_충전기_설치수  0.0571      1.8075        0.1033  0.1189
ev_200m     0.1370      1.7643        0.2417  0.2781
ev_등록대수     0.2664      1.9664        0.5239  0.6030

[ 최종 가중치 ]
  ev_충전기_설치수: 0.1189 (11.9%)
  ev_200m: 0.2781 (27.8%)
  ev_등록대수: 0.6030 (60.3%)


#### EV충전 그룹 CRITIC 한계

CRITIC 결과 **ev_등록대수(60.3%) >> ev_200m(27.8%) > ev_충전기_설치수(11.9%)** 로 산출되었으나,
태양광 직접 연계 변수인 ev_충전기_설치수가 가장 낮게 나온 것은 논리적으로 타당하지 않다.

**한계 원인: 집계 단위 불일치로 인한 표준편차 왜곡**

| 구분 | ev_충전기_설치수 | ev_200m | ev_등록대수 |
|------|--------------|---------|-----------|
| 집계 단위 | 주차장별 | 주차장별 | 자치구별 |
| 고유값 수 | 741개 | 741개 | 25개 |
| 정규화 후 표준편차 | 0.057 | 0.137 | 0.266 |

- ev_등록대수는 자치구별 단일값(25개)을 741개 주차장에 반복 매핑한 변수
- 자치구별 주차장 수가 불균등(영등포 75개 vs 서대문 11개)하여
  특정 등록대수 값이 쏠리는 구조
- 그 결과 MinMax 정규화 후 **표준편차가 인위적으로 크게 산출**
- CRITIC은 집계 단위 차이를 인식하지 못하므로
  자치구 단위 집계 변수가 포함될 경우 가중치가 과대평가됨
- ev_충전기_설치수는 실제 분포가 0에 극도로 쏠려(0개: 84.4%)
  표준편차가 낮게 나와 가중치가 과소평가됨

→ **집계단위가 혼재된 EV충전 그룹도 CRITIC 적용 불가**

---

### CRITIC 한계 요약

| 그룹 | 한계 원인 | 결론 |
|------|---------|------|
| 자체소비 | 이진변수(노외여부) 포함 → 표준편차 과대평가 | CRITIC 부적합 |
| EV충전 | 집계단위 불일치(자치구 vs 주차장별) → 표준편차 왜곡 | CRITIC 부적합 |

**→ 전체 가중치 산정을 상관관계 분석 + 민감도 분석으로 대체한다.**

---
### 4.2 자체소비 그룹 — 상관관계 분석 + 민감도 분석

In [15]:
# 상관관계 분석
cols_self = ['운영시간', '노외여부']
corr_self = df[cols_self].corr()

print("=== 자체소비 그룹 상관관계 ===")
print(corr_self.round(3))

print("\n=== 노상/노외별 운영시간 분포 ===")
print(df.groupby('노외여부')['운영시간'].describe().round(3))

=== 자체소비 그룹 상관관계 ===
       운영시간   노외여부
운영시간  1.000  0.676
노외여부  0.676  1.000

=== 노상/노외별 운영시간 분포 ===
      count    mean    std    min     25%     50%   75%   max
노외여부                                                         
0     209.0  10.746  2.737  7.857   8.857   9.143  13.0  24.0
1     532.0  20.714  5.519  7.143  17.000  24.000  24.0  24.0


In [16]:
# 노상 운영시간 확인
print("\n=== 노상 운영시간 분포 ===")
print(df[df['노외여부'] == 0]['운영시간'].value_counts().sort_index())

# 노외 운영시간 확인
print("\n=== 노외 운영시간 분포 ===")
print(df[df['노외여부'] == 1]['운영시간'].value_counts().sort_index())


=== 노상 운영시간 분포 ===
운영시간
7.857143      1
8.142857      1
8.285714      9
8.857143     90
9.000000      2
9.142857      3
9.428571      1
9.714286     10
10.000000    11
10.142857     4
10.714286     1
10.857143     3
11.000000     6
11.142857     1
11.285714     1
12.000000     9
12.714286     1
13.000000     4
13.285714     7
14.000000    24
14.714286    17
24.000000     3
Name: count, dtype: int64

=== 노외 운영시간 분포 ===
운영시간
7.142857       2
7.571429       1
8.142857       1
8.714286       1
8.857143       6
9.000000      12
9.142857       3
9.428571       1
9.500000       1
9.571429       1
9.857143       2
10.000000     28
10.857143      1
11.000000      9
11.285714      2
12.000000     10
12.571429      4
12.928571      3
13.000000      7
13.214286      1
13.285714      5
13.500000      1
13.642857      1
13.857143      1
14.000000     20
14.428571      1
14.714286      1
15.000000      4
16.142857      1
16.857143      1
17.000000      4
18.000000      2
18.285714      1
18.642857  

#### 상관관계 분석 결과

| 변수 쌍 | 상관계수 | 해석 |
|---------|---------|------|
| 운영시간 vs 노외여부 | 0.676 | 중간 상관 → 각각 독립적 정보 보유, 둘 다 유지 |

- r=0.676으로 상당한 상관이 있으나 완전하지 않아 두 변수 모두 유지
- r=0.676이지만 노상 24시간 운영 사례(3개) 등 서로를 완전히 대체하지 못함 → 둘 다 유지
- 노외 주차장이 상대적으로 24시간 운영 비중이 높고,
  노상 주차장이 시간 제한 운영 비중이 높은 구조가 반영된 결과
- 두 변수는 각각 주차장의 **시간적 사용 패턴**과 **공간·구조적 특성**을
  반영하므로 가중합 형태로 사용하는 것이 타당

In [17]:
# 민감도 분석 — 자체소비
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['운영시간_norm'] = scaler.fit_transform(df[['운영시간']])
df['노외여부_norm'] = df['노외여부'].astype(float)

scenarios_self = {
    '운영시간만 (1.0/0.0)': [1.0, 0.0],
    '운영시간 우선 (0.7/0.3)': [0.7, 0.3],
    '제안 (0.6/0.4)': [0.6, 0.4],
    '균등 (0.5/0.5)': [0.5, 0.5],
    '노외 우선 (0.3/0.7)': [0.3, 0.7],
}

ranks_self = {}
for name, weights in scenarios_self.items():
    score = (df['운영시간_norm'] * weights[0] +
             df['노외여부_norm'] * weights[1])
    ranks_self[name] = score.rank(ascending=False).astype(int)

rank_df_self = pd.DataFrame(ranks_self)
rank_df_self['pklt_nm'] = df['pklt_nm'].values
rank_df_self['노외여부'] = df['노외여부'].values
rank_df_self['운영시간'] = df['운영시간'].values

# 24시간 미만만 필터링 (변별력 확인)
non_24h = rank_df_self[rank_df_self['운영시간'] < 24].copy()
print(f"24시간 미만 주차장: {len(non_24h)}개")
print("\n=== 시나리오별 순위 변화 (상위 30개) ===")
print(non_24h.nsmallest(30, '제안 (0.6/0.4)').set_index('pklt_nm')[
    list(scenarios_self.keys()) + ['노외여부', '운영시간']
].to_string())

24시간 미만 주차장: 356개

=== 시나리오별 순위 변화 (상위 30개) ===
                  운영시간만 (1.0/0.0)  운영시간 우선 (0.7/0.3)  제안 (0.6/0.4)  균등 (0.5/0.5)  노외 우선 (0.3/0.7)  노외여부       운영시간
pklt_nm                                                                                                           
도림동 공영주차장                     386                383           383           383              383     1  23.000000
영등포구청별관 공영주차장                 386                383           383           383              383     1  23.000000
청계산 청룡주차장                     388                385           385           385              385     1  21.142857
대치2문화센터 공영주차장                 389                386           386           386              386     1  20.571429
일원1동(기계식)                     390                387           387           387              387     1  19.000000
개봉1동마을공동주차장                   393                390           390           390              390     1  18.642857
불암산 공영주차장                     39

#### 민감도 분석 결과 및 가중치 결정

> **24시간 미만 주차장(356개)만 분석 대상으로 한정**  
> 24시간 운영 주차장(385개, 52%)은 운영시간_norm=1.0으로 고정되어  
> 가중치가 바뀌어도 순위 변동이 없어 민감도 분석의 의미가 없음.  
> 운영시간이 다양한 주차장에서 가중치 변화에 따른 순위 안정성을 검증한다.


**결론: 운영시간 0.6 / 노외여부 0.4 채택**

| 시나리오 | 상위 30개 순위 변화 |
|---------|-----------------|
| 운영시간만 (1.0/0.0) | 상위권 일부 순위 변동 |
| 운영시간 우선 (0.7/0.3) | 이하 4개 시나리오와 동일 |
| **제안 (0.6/0.4)** | **← 채택** |
| 균등 (0.5/0.5) | 제안과 동일 |
| 노외 우선 (0.3/0.7) | 제안과 동일 |

**근거:**
1. 0.6/0.4 ~ 0.3/0.7 범위에서 상위 주차장 순위 완전히 안정적
   → 어느 가중치를 써도 결과가 동일하므로 순위 왜곡 없음
2. 운영시간은 연속형 변수로 더 많은 정보량을 포함
   → 이진형인 노외여부보다 소폭 높은 가중치 부여가 논리적으로 타당
3. 운영시간만(1.0/0.0) 시나리오에서만 순위 변동 발생
   → 노외여부를 완전히 배제하면 법적 설비 의무 차이를 반영 못 함

**→ 순위 안정성 + 변수 유형 논리를 동시에 만족하는 0.6/0.4 채택**

---
### 4.3 EV충전 그룹 — 상관관계 분석 + 민감도 분석

In [18]:
# 상관관계 분석
cols_ev = ['ev_충전기_설치수', 'ev_200m', 'ev_등록대수']
corr_ev = df[cols_ev].corr()

print("=== EV충전 그룹 상관관계 ===")
print(corr_ev.round(3))

=== EV충전 그룹 상관관계 ===
            ev_충전기_설치수  ev_200m  ev_등록대수
ev_충전기_설치수       1.000    0.197   -0.005
ev_200m          0.197    1.000    0.038
ev_등록대수         -0.005    0.038    1.000


#### 상관관계 분석 결과

| 변수 쌍 | 상관계수 | 해석 |
|---------|---------|------|
| ev_충전기_설치수 vs ev_200m | 0.197 | 약한 양의 상관 → 독립적 |
| ev_충전기_설치수 vs ev_등록대수 | -0.005 | 거의 독립 |
| ev_200m vs ev_등록대수 | 0.038 | 거의 독립 |

3개 변수 모두 r < 0.2로 다중공선성 없음 → 세 변수 모두 독립적인 정보를 보유

In [19]:
# 민감도 분석 — EV충전
ev_cols = ['ev_충전기_설치수', 'ev_200m', 'ev_등록대수']
df[['ev_내부_norm', 'ev_200m_norm', 'ev_등록_norm']] = scaler.fit_transform(df[ev_cols])

scenarios_ev = {
    '균등 (1/3씩)':          [1/3, 1/3, 1/3],
    '내부중심 (0.5/0.3/0.2)': [0.5, 0.3, 0.2],
    '제안 (0.4/0.3/0.3)':    [0.4, 0.3, 0.3],
    '근접중심 (0.2/0.5/0.3)': [0.2, 0.5, 0.3],
    '수요중심 (0.2/0.3/0.5)': [0.2, 0.3, 0.5],
}

ranks_ev = {}
for name, weights in scenarios_ev.items():
    score = (df['ev_내부_norm'] * weights[0] +
             df['ev_200m_norm'] * weights[1] +
             df['ev_등록_norm'] * weights[2])
    ranks_ev[name] = score.rank(ascending=False).astype(int)

rank_df_ev = pd.DataFrame(ranks_ev)
rank_df_ev['pklt_nm'] = df['pklt_nm'].values

top20 = rank_df_ev.nsmallest(20, '제안 (0.4/0.3/0.3)')
print("=== 상위 20개 시나리오별 순위 ===")
print(top20.set_index('pklt_nm').to_string())

=== 상위 20개 시나리오별 순위 ===
                균등 (1/3씩)  내부중심 (0.5/0.3/0.2)  제안 (0.4/0.3/0.3)  근접중심 (0.2/0.5/0.3)  수요중심 (0.2/0.3/0.5)
pklt_nm                                                                                                
영등포동제2공영                1                   1                 1                   9                  61
잠실역 공영주차장               2                   3                 2                   1                  29
수서역북 공영주차장              7                   5                 3                  17                   5
영희초교 공영주차장              4                  11                 5                   6                   2
일원1동(기계식)               4                  11                 5                   6                   2
일원동맛의거리                 4                  11                 5                   6                   2
탄천제2호 공영주차장             4                  11                 5                   6                   2
영등포동제3공영               31               

#### 민감도 분석 결과 및 가중치 결정

> **전체 741개 주차장 기준으로 분석**  
> EV충전 3개 변수 모두 연속형이므로 필터링 없이 전체 대상으로 순위 안정성 검증한다.

**결론: ev_충전기_설치수 0.4 / ev_200m 0.3 / ev_등록대수 0.3 채택**

| 시나리오 | 특징 | 순위 안정성 |
|---------|------|-----------|
| 균등 (1/3씩) | 세 변수 동등 반영 | 안정적 |
| 내부중심 (0.5/0.3/0.2) | 내부 충전기 과도 우위 | 안정적 |
| **제안 (0.4/0.3/0.3)** | **← 채택** | **가장 안정적** |
| 근접중심 (0.2/0.5/0.3) | 영등포제2공영 9위로 하락 | 불안정 |
| 수요중심 (0.2/0.3/0.5) | 영등포제2공영 61위, 영등포제3공영 97위로 폭락 | 불안정 |

**근거:**
1. 근접중심·수요중심에서 영등포동제2공영(내부 충전기 압도적 1위)이
   9위·61위로 폭락 → EV 수요 편중 시 직접 연계 변수 과소평가
2. 수요중심에서 영등포동제3공영 97위, 천호역 91위
   → ev_등록대수 단독 강조 시 실제 충전 인프라와 역전 현상 발생
3. 균등~제안 범위(내부 0.33~0.4)에서 상위권 순위 안정적
   → ev_충전기_설치수가 태양광 직접 연계 변수로서 소폭 우위 부여가 타당

**→ 직접 연계 변수 우위 + 순위 안정성을 동시에 만족하는 0.4/0.3/0.3 채택**

---
### 4.4 하위축 간 가중치 — CRITIC

자체소비 점수와 EV충전 점수는 모두 연속형이며 주차장별 개별값으로
집계단위가 동일하여 CRITIC 적용이 가능하다.

> 기존에는 대한전기학회 논문(자체소비 78.8% / EV충전 21.2%)을 근거로
> 0.8 / 0.2를 적용하였으나, 해당 논문의 79%는 '자체소비(조명·설비)'가 아닌
> **'ESS 저장용 전력'** 으로 확인되어 논문 근거를 폐기한다.
> 하위축 간 가중치는 데이터 기반 CRITIC으로 결정한다.

In [22]:
# ── CRITIC 투입용 점수 산출 ──────────────────────────
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

df['운영시간_norm'] = scaler.fit_transform(df[['운영시간']])
df['노외여부_norm'] = df['노외여부'].astype(float)
df['자체소비_점수'] = df['운영시간_norm'] * 0.6 + df['노외여부_norm'] * 0.4

ev_cols = ['ev_충전기_설치수', 'ev_200m', 'ev_등록대수']
df[['ev_내부_norm', 'ev_200m_norm', 'ev_등록_norm']] = scaler.fit_transform(df[ev_cols])
df['EV충전_점수'] = (df['ev_내부_norm'] * 0.4 +
                    df['ev_200m_norm'] * 0.3 +
                    df['ev_등록_norm'] * 0.3)
df['EV충전_점수_norm'] = scaler.fit_transform(df[['EV충전_점수']])

In [23]:
# ── CRITIC ───────────────────────────────────────────
cols_axis = ['자체소비_점수', 'EV충전_점수_norm']
weights_axis, C_axis, std_axis, corr_axis = critic_weights(df[cols_axis])

print("=== 하위축 간 CRITIC 결과 ===\n")
print("[ 상관행렬 ]")
print(corr_axis.round(3))
print("\n[ 변수별 지표 ]")
summary_axis = pd.DataFrame({
    '표준편차': std_axis.round(4),
    '독립성합산(1-r)': C_axis.div(std_axis).round(4),
    'CRITIC 지수(C)': C_axis.round(4),
    '가중치': weights_axis.round(4)
})
print(summary_axis.to_string())
print(f"\n[ 최종 가중치 ]")
for col, w in weights_axis.items():
    print(f"  {col}: {w:.4f} ({w*100:.1f}%)")

=== 하위축 간 CRITIC 결과 ===

[ 상관행렬 ]
              자체소비_점수  EV충전_점수_norm
자체소비_점수         1.000        -0.012
EV충전_점수_norm   -0.012         1.000

[ 변수별 지표 ]
                표준편차  독립성합산(1-r)  CRITIC 지수(C)     가중치
자체소비_점수       0.3919      1.0124        0.3968  0.6958
EV충전_점수_norm  0.1714      1.0124        0.1735  0.3042

[ 최종 가중치 ]
  자체소비_점수: 0.6958 (69.6%)
  EV충전_점수_norm: 0.3042 (30.4%)


#### CRITIC 결과

| 지표 | 자체소비_점수 | EV충전_점수_norm |
|------|------------|----------------|
| 표준편차 | 0.392 | 0.171 |
| 독립성 합산(1-r) | 1.012 | 1.012 |
| CRITIC 지수 | 0.397 | 0.174 |
| **가중치** | **0.696 (69.6%)** | **0.304 (30.4%)** |

#### 결과 해석

- 상관계수 -0.012 → 두 하위축 거의 완전 독립 → 독립성 합산값 동일
- 가중치는 표준편차(변별력)에 의해 결정
- 자체소비_점수의 표준편차(0.392)가 EV충전_점수_norm(0.171)의 2.3배
  → 주차장 간 자체소비 차이가 EV충전 차이보다 훨씬 크다는 데이터 반영
- 두 변수 모두 연속형 + 주차장별 개별값으로 집계단위 동일
  → 이진변수/집계단위 문제 없음, CRITIC 결과 신뢰 가능

**→ CRITIC 산출값 반올림: 자체소비 0.7 / EV충전 0.3 채택**

In [24]:
# ── 민감도 분석 ───────────────────
scenarios_axis = {
    '자체소비 중심 (0.8/0.2)': [0.8, 0.2],
    '자체소비 우선 (0.7/0.3)': [0.7, 0.3],
    '균등 (0.5/0.5)':          [0.5, 0.5],
    'EV충전 중심 (0.4/0.6)':   [0.4, 0.6],
}

print("=== 하위축 간 가중치 민감도 분석 ===\n")
for name, weights in scenarios_axis.items():
    score = (df['자체소비_점수'] * weights[0] +
             df['EV충전_점수_norm'] * weights[1])
    ones  = (score == 1.0).sum()
    median = score.median()
    std    = score.std()
    print(f"{name} → 1.0: {ones}개, 중앙값: {median:.3f}, std: {std:.3f}")

=== 하위축 간 가중치 민감도 분석 ===

자체소비 중심 (0.8/0.2) → 1.0: 1개, 중앙값: 0.801, std: 0.307
자체소비 우선 (0.7/0.3) → 1.0: 1개, 중앙값: 0.703, std: 0.272
균등 (0.5/0.5) → 1.0: 1개, 중앙값: 0.516, std: 0.208
EV충전 중심 (0.4/0.6) → 1.0: 1개, 중앙값: 0.425, std: 0.183


#### 민감도 분석 교차 검증

| 시나리오 | std | 중앙값 |
|---------|-----|--------|
| 자체소비 중심 (0.8/0.2) | 0.307 | 0.801 |
| **자체소비 우선 (0.7/0.3)** | **0.272** | **0.703** |
| 균등 (0.5/0.5) | 0.208 | 0.516 |
| EV충전 중심 (0.4/0.6) | 0.183 | 0.425 |

- std 최대는 0.8/0.2이나, 이는 자체소비가 과도하게 지배하는 구간
- CRITIC 산출값(0.696/0.304)과 가장 근접한 0.7/0.3이
  데이터 기반 근거와 변별력을 동시에 만족

**→ CRITIC 결과(0.7/0.3) 최종 채택**

### 4.5 최종 가중치 요약

| 단계 | 변수 | 가중치 | 산정 방법 | 근거 |
|------|------|--------|---------|------|
| 자체소비 점수 | 운영시간 | 0.6 | 상관관계+민감도 | 연속형 변수, 더 많은 정보량 |
| 자체소비 점수 | 노외여부 | 0.4 | 상관관계+민감도 | 주차장법 상시 조명 의무 |
| EV충전 점수 | ev_충전기_설치수 | 0.4 | 상관관계+민감도 | 태양광 직접 연계 가장 확실 |
| EV충전 점수 | ev_200m | 0.3 | 상관관계+민감도 | 도보 2~3분 이내 인근 인프라 |
| EV충전 점수 | ev_등록대수 | 0.3 | 상관관계+민감도 | 잠재 EV 충전 수요 |
| **축3 최종** | 자체소비 점수 | **0.7** | **CRITIC** | 변별력 2.3배 높음, 연속형+동일 집계단위 |
| **축3 최종** | EV충전 점수 | **0.3** | **CRITIC** | 변별력 상대적 낮음 |

> **방법론 선택 요약**
> - 자체소비/EV충전 그룹 내부: 이진변수·집계단위 문제로 CRITIC 부적합
>   → 상관관계 분석 + 민감도 분석으로 대체
> - 하위축 간: 연속형 + 동일 집계단위로 CRITIC 적합
>   → CRITIC 결과 채택

---
## 5. 축3 점수 산출

### 5.1 점수화 공식

$$운영시간 = \frac{평일운영시간 \times 5 + 주말운영시간 \times 2}{7}$$

$$자체소비\_점수 = MinMax(운영시간) \times 0.6 + 노외여부 \times 0.4$$

$$EV충전\_점수 = MinMax(ev\_충전기\_설치수) \times 0.4 + MinMax(ev\_200m) \times 0.3 + MinMax(ev\_등록대수) \times 0.3$$

$$EV충전\_점수\_norm = MinMax(EV충전\_점수)$$

$$축3\_자가소비적합성 = 자체소비\_점수 \times 0.7 + EV충전\_점수\_norm \times 0.3$$

#### EV충전 점수 재정규화 이유

EV충전_점수는 세 변수를 MinMax 정규화 후 가중합하여 산출하지만,
가중합 결과가 반드시 0~1 범위를 채우지는 않는다.

**원인**: 세 변수(ev_충전기_설치수, ev_200m, ev_등록대수)가
동시에 최댓값인 주차장이 존재하지 않을 수 있어,
실제 EV충전_점수의 최댓값이 이론적 최대(1.0)보다 작게 형성됨

예시:
- ev_충전기_설치수 최대 주차장 ≠ ev_200m 최대 주차장 ≠ ev_등록대수 최대 주차장
- 실제 EV충전_점수 분포: 0 ~ 0.56 수준

**문제**: 재정규화 없이 자체소비_점수(0~1)와 합산하면
EV충전의 실질 기여가 의도한 가중치(0.3)보다 축소됨

**해결**: EV충전_점수를 MinMax 재정규화(0~1)하여
두 하위축 점수의 스케일을 통일한 후 0.7/0.3으로 합산

> **한계**: 재정규화 과정에서 EV충전 점수 내부의 절대적 차이가
> 상대적 차이로 변환되어 점수 간 격차가 과장될 수 있음.
> 두 하위축의 스케일 통일을 위한 불가피한 처리로 명시한다.

In [25]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# ── 자체소비 점수 ────────────────────────────────────────────────────────
df['운영시간_norm'] = scaler.fit_transform(df[['운영시간']])
df['노외여부_norm'] = df['노외여부'].astype(float)
df['자체소비_점수'] = df['운영시간_norm'] * 0.6 + df['노외여부_norm'] * 0.4

# ── EV충전 점수 ──────────────────────────────────────────────────────────
ev_cols = ['ev_충전기_설치수', 'ev_200m', 'ev_등록대수']
df[['ev_내부_norm', 'ev_200m_norm', 'ev_등록_norm']] = scaler.fit_transform(df[ev_cols])
df['EV충전_점수'] = (df['ev_내부_norm'] * 0.4 +
                    df['ev_200m_norm'] * 0.3 +
                    df['ev_등록_norm'] * 0.3)
df['EV충전_점수_norm'] = scaler.fit_transform(df[['EV충전_점수']])

# ── 축3 최종 점수 ────────────────────────────────────────────────────────
df['축3_자가소비적합성'] = df['자체소비_점수'] * 0.7 + df['EV충전_점수_norm'] * 0.3

print(df[['pklt_nm', 'pklt_knd_nm', '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성']]
      .sort_values('축3_자가소비적합성', ascending=False)
      .head(15).to_string())

print(f"\n축3 점수 분포:")
print(df['축3_자가소비적합성'].describe().round(3))

            pklt_nm pklt_knd_nm  자체소비_점수  EV충전_점수_norm  축3_자가소비적합성
521        영등포동제2공영      노외 주차장      1.0      1.000000    1.000000
587       잠실역 공영주차장      노외 주차장      1.0      0.771839    0.931552
414      수서역북 공영주차장      노외 주차장      1.0      0.725189    0.917557
664     탄천제2호 공영주차장      노외 주차장      1.0      0.703662    0.911099
528      영희초교 공영주차장      노외 주차장      1.0      0.703662    0.911099
522        영등포동제3공영      노외 주차장      1.0      0.685415    0.905624
499   역삼1문화센터 공영주차장      노외 주차장      1.0      0.665387    0.899616
501  역삼문화공원제1호공영주차장      노외 주차장      1.0      0.665387    0.899616
500    역삼문화공원 공영주차장      노외 주차장      1.0      0.665387    0.899616
188  도곡로21길 7 공영주차장      노외 주차장      1.0      0.665387    0.899616
186   도곡로 327 공영주차장      노외 주차장      1.0      0.665387    0.899616
147      논현초교 공영주차장      노외 주차장      1.0      0.621224    0.886367
680       학여울 공영주차장      노외 주차장      1.0      0.611885    0.883566
645       천호역 공영주차장      노외 주차장      1.0      0.603446    0.88

### 5.2 점수 분포 요약

In [26]:
# 점수 분포 상세
print("=== 자체소비 점수 분포 ===")
print(df['자체소비_점수'].describe().round(3))
print(f"1.0인 주차장: {(df['자체소비_점수']==1.0).sum()}개 ({(df['자체소비_점수']==1.0).mean()*100:.1f}%)")
print(f"0.5 미만: {(df['자체소비_점수']<0.5).sum()}개 ({(df['자체소비_점수']<0.5).mean()*100:.1f}%)")

print("\n=== EV충전 점수(정규화) 분포 ===")
print(df['EV충전_점수_norm'].describe().round(3))
print(f"0.5 이상: {(df['EV충전_점수_norm']>=0.5).sum()}개 ({(df['EV충전_점수_norm']>=0.5).mean()*100:.1f}%)")
print(f"0.1 미만: {(df['EV충전_점수_norm']<0.1).sum()}개 ({(df['EV충전_점수_norm']<0.1).mean()*100:.1f}%)")

print("\n=== 축3 최종 점수 분포 ===")
print(df['축3_자가소비적합성'].describe().round(3))

=== 자체소비 점수 분포 ===
count    741.000
mean       0.670
std        0.382
min        0.025
25%        0.244
50%        1.000
75%        1.000
max        1.000
Name: 자체소비_점수, dtype: float64
1.0인 주차장: 382개 (51.6%)
0.5 미만: 237개 (32.0%)

=== EV충전 점수(정규화) 분포 ===
count    741.000
mean       0.199
std        0.171
min        0.000
25%        0.069
50%        0.147
75%        0.268
max        1.000
Name: EV충전_점수_norm, dtype: float64
0.5 이상: 76개 (10.3%)
0.1 미만: 278개 (37.5%)

=== 축3 최종 점수 분포 ===
count    741.000
mean       0.529
std        0.272
min        0.029
25%        0.309
50%        0.703
75%        0.744
max        1.000
Name: 축3_자가소비적합성, dtype: float64


#### 점수 분포 요약

| 통계량 | 자체소비 점수 | EV충전 점수(정규화) | 축3 최종 점수 |
|--------|------------|-----------------|------------|
| 평균 | 0.670 | 0.199 | 0.529 |
| 표준편차 | 0.382 | 0.171 | 0.272 |
| 중앙값 | 1.000 | 0.147 | 0.703 |
| 최댓값 | 1.000 | 1.000 | 1.000 |

### 5.3 결과 저장

In [27]:
output_cols = ['pklt_nm', 'pklt_knd_nm', 'lat', 'lot', '자치구',
               '운영시간', '노외여부',
               'ev_충전기_설치수', 'ev_200m', 'ev_등록대수',
               '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성']

df_result = df[output_cols].copy()
df_result.to_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\parking_axis3_scored.csv',
    index=False,
    encoding='utf-8-sig'
)
print(f"저장 완료: {len(df_result)}개")
print(f"\n축3 점수 분포:")
print(df_result['축3_자가소비적합성'].describe().round(3))

저장 완료: 741개

축3 점수 분포:
count    741.000
mean       0.529
std        0.272
min        0.029
25%        0.309
50%        0.703
75%        0.744
max        1.000
Name: 축3_자가소비적합성, dtype: float64


---
## 6. 시각화 및 인사이트

In [28]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 6.1 축3 점수 지도 ────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scattermap(
    lat=df_result['lat'],
    lon=df_result['lot'],
    mode='markers',
    marker=dict(
        size=8,
        color=df_result['축3_자가소비적합성'],
        colorscale='Blues',
        showscale=True,
        colorbar=dict(title='축3 점수'),
        cmin=0, cmax=1
    ),
    text=(df_result['pklt_nm'] + '<br>' +
          '자치구: ' + df_result['자치구'] + '<br>' +
          '축3 점수: ' + df_result['축3_자가소비적합성'].round(3).astype(str) + '<br>' +
          '자체소비: ' + df_result['자체소비_점수'].round(3).astype(str) + '<br>' +
          'EV 내부(50m): ' + df_result['ev_충전기_설치수'].astype(str) + '개<br>' +
          'EV 근접(200m): ' + df_result['ev_200m'].astype(str) + '개'),
    hovertemplate='%{text}<extra></extra>'
))
fig.update_layout(
    map=dict(style='carto-positron',
             center=dict(lat=37.5665, lon=126.9780), zoom=10),
    title='축3 자가소비 적합성 점수 지도 (741개) — 최종',
    height=600
)
fig.show()
fig.write_html(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\figures\축3_점수_지도_최종.html'
)

#### 6.1 축3 점수 지도 인사이트

- 고득점(진한 파랑) 주차장이 **도심(강남·영등포·송파) 및 한강 이북 중심부**에 집중
- 외곽(노원·도봉·금천·강서 일부)은 연한 색 → 상대적으로 낮은 자가소비 적합성
- 한강 이남 강남·서초 일대에 고득점 주차장이 밀집 →
  24시간 노외 운영 + EV 등록대수 최상위 자치구 효과가 복합 반영된 결과

### 6.2 자체소비 vs EV충전 산점도 (4분면 분석)

In [29]:
# ── 6.2 자체소비 vs EV충전 산점도 (4분면) ────────────────────────────────
mid_x = df_result['자체소비_점수'].median()
mid_y = df_result['EV충전_점수_norm'].median()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=df_result['자체소비_점수'],
    y=df_result['EV충전_점수_norm'],
    mode='markers',
    marker=dict(
        size=6,
        color=df_result['축3_자가소비적합성'],
        colorscale='Blues',
        showscale=True,
        colorbar=dict(title='축3 점수'),
    ),
    text=(df_result['pklt_nm'] + '<br>' +
          '자체소비: ' + df_result['자체소비_점수'].round(3).astype(str) + '<br>' +
          'EV충전: ' + df_result['EV충전_점수_norm'].round(3).astype(str)),
    hovertemplate='%{text}<extra></extra>'
))
fig2.add_hline(y=mid_y, line_dash='dash', line_color='gray', opacity=0.5)
fig2.add_vline(x=mid_x, line_dash='dash', line_color='gray', opacity=0.5)

fig2.add_annotation(x=0.9, y=0.9,  text='★ 최우선<br>(자체소비↑ EV↑)', showarrow=False, font=dict(size=11))
fig2.add_annotation(x=0.1, y=0.9,  text='EV충전 강점<br>(자체소비↓ EV↑)', showarrow=False, font=dict(size=11))
fig2.add_annotation(x=0.9, y=0.05, text='자체소비 강점<br>(자체소비↑ EV↓)', showarrow=False, font=dict(size=11))
fig2.add_annotation(x=0.1, y=0.05, text='낮은 적합성<br>(자체소비↓ EV↓)', showarrow=False, font=dict(size=11))

fig2.update_layout(
    title='자체소비 점수 vs EV충전 점수 산점도',
    xaxis_title='자체소비_점수',
    yaxis_title='EV충전_점수_norm',
    height=500
)
fig2.show()
fig2.write_html(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\figures\자체소비_vs_EV충전_산점도.html'
)

#### 4분면 기준선 설정 — 평균 채택

4분면 구분 기준으로 중앙값을 먼저 시도하였으나,
자체소비_점수의 중앙값이 1.0으로 확인되어 기준선으로 사용할 수 없었다.

**원인**: 24시간 노외 주차장(382개, 51.6%)이 자체소비_점수 1.0으로 수렴하여
중앙값이 1.0으로 형성됨 → 중앙값 기준 시 대부분의 주차장이 '자소↓'로 분류되는 왜곡 발생

**→ 평균을 기준선으로 채택**
- 자체소비_점수 평균: 0.670
- EV충전_점수_norm 평균: 0.199
- 평균 기준은 분포의 쏠림에 상대적으로 덜 민감하며,
  "평균 이상 / 이하"로 직관적으로 해석 가능

In [45]:
import plotly.graph_objects as go

mid_x = df_result['자체소비_점수'].mean()
mid_y = df_result['EV충전_점수_norm'].mean()

fig2_1 = go.Figure()

# ── 4분면 배경색 ──────────────────────────────────────
quad_colors = [
    dict(x0=mid_x, x1=1.02, y0=mid_y, y1=1.02, color='rgba(31,78,121,0.08)'),   # 최우선 (우상)
    dict(x0=-0.02, x1=mid_x, y0=mid_y, y1=1.02, color='rgba(46,117,182,0.06)'),  # EV강점 (좌상)
    dict(x0=mid_x, x1=1.02, y0=-0.02, y1=mid_y, color='rgba(112,173,71,0.08)'),  # 자소강점 (우하)
    dict(x0=-0.02, x1=mid_x, y0=-0.02, y1=mid_y, color='rgba(180,180,180,0.08)'),# 낮은적합 (좌하)
]
for q in quad_colors:
    fig2_1.add_shape(type='rect',
        x0=q['x0'], x1=q['x1'], y0=q['y0'], y1=q['y1'],
        fillcolor=q['color'], line=dict(width=0), layer='below'
    )

# ── 분면별 산점도 ─────────────────────────────────────
color_map = {
    '최우선 (자소↑EV↑)':   '#1F4E79',
    'EV강점 (자소↓EV↑)':   '#2E75B6',
    '자소강점 (자소↑EV↓)':  '#70AD47',
    '낮은적합 (자소↓EV↓)':  '#AAAAAA',
}

for 분면, color in color_map.items():
    subset = df_result[df_result['분면'] == 분면]
    fig2_1.add_trace(go.Scatter(
        x=subset['자체소비_점수'],
        y=subset['EV충전_점수_norm'],
        mode='markers',
        name=f"{분면} ({len(subset)}개)",
        marker=dict(size=7, color=color, opacity=0.75,
                    line=dict(width=0.5, color='white')),
        text=(subset['pklt_nm'] + '<br>' +
              '자체소비: ' + subset['자체소비_점수'].round(3).astype(str) + '<br>' +
              'EV충전: '  + subset['EV충전_점수_norm'].round(3).astype(str) + '<br>' +
              '축3 점수: ' + subset['축3_자가소비적합성'].round(3).astype(str)),
        hovertemplate='%{text}<extra></extra>'
    ))

# ── 기준선 ────────────────────────────────────────────
fig2_1.add_vline(x=mid_x, line_dash='dash', line_color='#888888', line_width=1.5,
              opacity=0.7)
fig2_1.add_hline(y=mid_y, line_dash='dash', line_color='#888888', line_width=1.5,
              opacity=0.7)

# ── 4분면 레이블 ──────────────────────────────────────
labels = [
    dict(x=0.95, y=0.95, text='★ 최우선<br>(자소↑ EV↑)', color='#1F4E79', xa='right', ya='top'),
    dict(x=0.05, y=0.95, text='EV충전 강점<br>(자소↓ EV↑)', color='#2E75B6', xa='left', ya='top'),
    dict(x=0.95, y=0.05, text='자체소비 강점<br>(자소↑ EV↓)', color='#538135', xa='right', ya='bottom'),
    dict(x=0.05, y=0.05, text='낮은 적합성<br>(자소↓ EV↓)', color='#888888', xa='left', ya='bottom'),
]
for lb in labels:
    fig2_1.add_annotation(
        xref='paper', yref='paper',
        x=lb['x'], y=lb['y'],
        text=lb['text'],
        showarrow=False,
        font=dict(size=11, color=lb['color']),
        xanchor=lb['xa'], yanchor=lb['ya'],
        bgcolor='rgba(255,255,255,0.6)',
        borderpad=4
    )

# ── 평균 기준선 레이블 ────────────────────────────────
fig2_1.add_annotation(
    x=mid_x, y=1.04, xref='x', yref='paper',
    text=f"자체소비 평균 {mid_x:.3f}",
    showarrow=False, font=dict(size=10, color='#888888'), xanchor='left'
)
fig2_1.add_annotation(
    x=1.04, y=mid_y, xref='paper', yref='y',
    text=f"EV충전 평균 {mid_y:.3f}",
    showarrow=False, font=dict(size=10, color='#888888'), xanchor='left'
)

fig2_1.update_layout(
    title=dict(text='자체소비 점수 vs EV충전 점수 산점도', font=dict(size=16)),
    xaxis=dict(title='자체소비_점수', range=[-0.02, 1.05],
               showgrid=True, gridcolor='#E8E8E8', zeroline=False),
    yaxis=dict(title='EV충전_점수_norm', range=[-0.02, 1.05],
               showgrid=True, gridcolor='#E8E8E8', zeroline=False),
    legend=dict(
    x=1.02,
    y=1.0,
    xanchor='left',
    yanchor='top',
    bgcolor='rgba(255,255,255,0.85)',
    bordercolor='#CCCCCC',
    borderwidth=1
),
    plot_bgcolor='#F8FBFD',
    paper_bgcolor='white',
    height=560,
    width=800,
    margin=dict(r=250)
)

fig2_1.show()
fig2_1.write_html(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\figures\자체소비_vs_EV충전_산점도_평균기준.html'
)

In [48]:
# 특정 주차장 분면 확인
target = '가산디지털 서측'
print(df_result[df_result['pklt_nm'].str.contains(target)][
    ['pklt_nm', '자체소비_점수', 'EV충전_점수_norm', '분면']
].to_string())

     pklt_nm   자체소비_점수  EV충전_점수_norm             분면
19  가산디지털 서측  0.669492      0.133553  낮은적합 (자소↓EV↓)


In [43]:
# 4분면 기준선 — 평균
mid_x = df_result['자체소비_점수'].mean()
mid_y = df_result['EV충전_점수_norm'].mean()

print(f"자체소비_점수 평균: {mid_x:.3f}")
print(f"EV충전_점수_norm 평균: {mid_y:.3f}")

# 4분면 분류
df_result['분면'] = df_result.apply(lambda r:
    '최우선 (자소↑EV↑)'  if r['자체소비_점수'] >= mid_x and r['EV충전_점수_norm'] >= mid_y else
    'EV강점 (자소↓EV↑)'  if r['자체소비_점수'] <  mid_x and r['EV충전_점수_norm'] >= mid_y else
    '자소강점 (자소↑EV↓)' if r['자체소비_점수'] >= mid_x and r['EV충전_점수_norm'] <  mid_y else
    '낮은적합 (자소↓EV↓)', axis=1
)

# 분면별 주차장 수
print("\n=== 분면별 주차장 수 ===")
print(df_result['분면'].value_counts())

# 분면별 대표 주차장 (축3 점수 상위 3개)
print("\n=== 분면별 대표 주차장 ===")
for q in ['최우선 (자소↑EV↑)', 'EV강점 (자소↓EV↑)', '자소강점 (자소↑EV↓)', '낮은적합 (자소↓EV↓)']:
    top = df_result[df_result['분면'] == q].nlargest(3, '축3_자가소비적합성')
    print(f"\n{q}")
    print(top[['pklt_nm', '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성']].to_string())

자체소비_점수 평균: 0.670
EV충전_점수_norm 평균: 0.199

=== 분면별 주차장 수 ===
분면
자소강점 (자소↑EV↓)    254
낮은적합 (자소↓EV↓)    206
최우선 (자소↑EV↑)     152
EV강점 (자소↓EV↑)    129
Name: count, dtype: int64

=== 분면별 대표 주차장 ===

최우선 (자소↑EV↑)
        pklt_nm  자체소비_점수  EV충전_점수_norm  축3_자가소비적합성
521    영등포동제2공영      1.0      1.000000    1.000000
587   잠실역 공영주차장      1.0      0.771839    0.931552
414  수서역북 공영주차장      1.0      0.725189    0.917557

EV강점 (자소↓EV↑)
              pklt_nm   자체소비_점수  EV충전_점수_norm  축3_자가소비적합성
36            강남한티주차장  0.618644      0.665387    0.632667
196  도산대로25길 32 공영주차장  0.644068      0.547620    0.615133
239        마곡8 임시 주차장  0.644068      0.439477    0.582690

자소강점 (자소↑EV↓)
           pklt_nm  자체소비_점수  EV충전_점수_norm  축3_자가소비적합성
366  서강대역 환승 공영주차장      1.0      0.187769    0.756331
443     신월4동 공영주차장      1.0      0.187561    0.756268
183        대흥공영주차장      1.0      0.184825    0.755448

낮은적합 (자소↓EV↓)
            pklt_nm   자체소비_점수  EV충전_점수_norm  축3_자가소비적합성
19         가산디지털 서측  0.669492      0.133

#### 6.2 자체소비 vs EV충전 산점도 인사이트

**4분면 기준선**: 자체소비 평균 0.670 / EV충전 평균 0.199

*(중앙값은 자체소비가 1.0으로 기준선 역할 불가 → 평균 채택)*

**4분면 분포 요약**

| 분면 | 주차장 수 | 대표 주차장 | 특징 |
|------|---------|-----------|------|
| 최우선 (우상단, 자소↑ EV↑) | 152개 | 영등포동제2공영, 잠실역, 수서역북 | 두 하위축 모두 평균 이상 → 진정한 최우선 입지 |
| EV충전 강점 (좌상단, 자소↓ EV↑) | 129개 | 강남한티주차장, 도산대로25길 | EV 수요 높은 지역 입지, 자체소비 상대적 낮음 |
| 자체소비 강점 (우하단, 자소↑ EV↓) | 254개 | 서강대역 환승, 신월4동, 대흥공영 | 24시간 노외 다수, EV 인프라 부족으로 순위 하락 |
| 낮은 적합성 (좌하단, 자소↓ EV↓) | 206개 | 가산디지털 서측, 도봉산 | 단기 운영 + EV 인프라 미비 → 우선순위 후순위 |

- **자소강점(254개)이 가장 많은 이유**: x=1.0에 수렴한 24시간 노외 주차장 382개 중
  EV충전 평균(0.199) 이하인 주차장이 대다수 → EV 인프라 보급이 아직 부족한 현실 반영
- **EV충전이 실질적 변별 요소**: 자체소비 1.0으로 수렴한 구간에서 최종 순위는 EV충전 점수가 결정
- **EV충전 점수 평균(0.199) 이하에 전체의 약 62%가 분포** →
  서울시 공영주차장 EV 충전 인프라가 전반적으로 부족함을 데이터로 확인

**선행연구와의 비교 검증**

심혜영 (2022), "서울시 전기차 보급을 위한 공공 급속충전기의 최적입지 분석"에 따르면
영등포구가 공공 급속충전기 35개(자치구 중 1위),
강남구 31개(2위)로 EV 충전 인프라가 집중되어 있는 것으로 확인되었다.

본 분석의 50m 프록시 결과에서도
영등포동제2공영(ev_충전기_설치수 62개)이 전체 1위,
강남구 주차장이 Top5 내 다수를 차지하는 것과 일치한다.

→ 선행연구와 방향성이 일치하여 50m 프록시의 상대적 타당성을 간접 확인

### 6.3 자치구별 평균 축3 점수

- 상위: 성동구(0.805), 성북구(0.802), 은평구(0.794)
- 하위: 영등포구(0.343), 노원구(0.347), 금천구(0.354)
- 강남구(0.626) 중간 — EV 등록 1위지만 노상 주차장 비율 영향

In [35]:
# ── 6.3 자치구별 평균 축3 점수 ───────────────────────────────────────────
gu_score = (df_result.groupby('자치구')['축3_자가소비적합성']
            .mean()
            .sort_values(ascending=False)
            .round(3))

print("자치구별 평균 축3 점수:")
print(gu_score.to_string())

fig3 = go.Figure(go.Bar(
    x=gu_score.index,
    y=gu_score.values,
    marker_color=gu_score.values,
    marker_colorscale='Blues',
    text=gu_score.values.round(3),
    textposition='outside'
))
fig3.update_layout(
    title='자치구별 평균 축3 자가소비 적합성 점수',
    xaxis_title='자치구',
    yaxis_title='평균 축3 점수',
    height=500
)
fig3.show()
fig3.write_html(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\figures\자치구별_축3_점수.html'
)

자치구별 평균 축3 점수:
자치구
성동구     0.728
서초구     0.725
성북구     0.716
은평구     0.715
중랑구     0.675
송파구     0.648
광진구     0.639
강동구     0.629
강남구     0.622
용산구     0.616
동작구     0.595
구로구     0.590
동대문구    0.584
강서구     0.531
마포구     0.502
강북구     0.492
관악구     0.489
종로구     0.489
중구      0.480
도봉구     0.471
양천구     0.468
서대문구    0.360
영등포구    0.327
금천구     0.326
노원구     0.325


#### 6.3 자치구별 평균 축3 점수 인사이트

**상위 자치구**

| 순위 | 자치구 | 평균 점수 | 특징 |
|------|---------|---------|------|
| 1 | 성동구 | 0.728 | 노외 주차장 비중 높고 24시간 운영 다수, 한양대·성수 일대 EV 인프라 양호 |
| 2 | 서초구 | 0.725 | EV 등록대수 상위권 자치구, 24시간 노외 주차장 집중 |
| 3 | 성북구 | 0.716 | 노외 주차장 비중 높고 24시간 운영 비중이 커 자체소비 점수 높게 수렴 |
| 4 | 은평구 | 0.715 | 주차장 수(16개)가 적어 24시간 노외 위주로 구성되어 평균이 높게 형성 |

**하위 자치구**

| 순위 | 자치구 | 평균 점수 | 특징 |
|------|---------|---------|------|
| 22 | 금천구 | 0.326 | 노상 주차장 비중 높고 단기 운영 집중, EV 등록대수 하위권 |
| 23 | 영등포구 | 0.327 | 주차장 수(75개) 최다이나 노상 단기 운영 다수로 평균 희석, 개별 주차장(영등포동제2·3공영)은 최상위권 |
| 24 | 노원구 | 0.325 | 외곽 입지로 EV 등록대수 낮고 노상·단기 운영 주차장 비중 높음 |
| 25 | 서대문구 | 0.360 | 주차장 수(11개) 최소, 소규모 노상 주차장 위주로 EV 충전 인프라 미비 |

> **영등포구**: 개별 주차장(영등포동제2·3공영) 최상위권이나  
> 노상 주차장이 많아 자치구 평균은 하위권 — 자치구 평균보다 개별 입지 분석이 중요함을 시사

### 6.4 노상/노외별 점수 분포

- 노외 중앙값 0.80 vs 노상 중앙값 0.15 → 명확한 차이
- 노외여부 가중치(0.4) 부여가 실질적으로 유의미한 차이를 만들어냄

In [39]:
# ── 6.4 노상/노외별 점수 분포 박스플롯 ──────────────────────────────────
df_result['주차장유형'] = df_result['pklt_knd_nm'].apply(
    lambda x: '노외 주차장' if '노외' in str(x) else '노상 주차장'
)

fig4 = go.Figure()

for 유형, color in [('노외 주차장', '#1f77b4'), ('노상 주차장', '#aec7e8')]:
    subset = df_result[df_result['주차장유형'] == 유형]
    fig4.add_trace(go.Box(
        y=subset['축3_자가소비적합성'],
        name=f"{유형} ({len(subset)}개)",
        marker_color=color,
        boxmean='sd',  # 평균 + 표준편차 표시
        hovertemplate=(
            '유형: ' + 유형 + '<br>' +
            '점수: %{y:.3f}<extra></extra>'
        )
    ))

fig4.update_layout(
    title='노상/노외별 축3 자가소비 적합성 점수 분포',
    yaxis_title='축3 자가소비적합성 점수',
    height=500,
    showlegend=True
)
fig4.show()
fig4.write_html(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\figures\노상노외_축3_박스플롯.html'
)

# 통계 요약
print("=== 노상/노외별 축3 점수 통계 ===")
print(df_result.groupby('주차장유형')['축3_자가소비적합성'].describe().round(3).to_string())

=== 노상/노외별 축3 점수 통계 ===
        count   mean    std    min    25%    50%    75%   max
주차장유형                                                        
노상 주차장  209.0  0.155  0.093  0.029  0.089  0.115  0.218  0.47
노외 주차장  532.0  0.676  0.151  0.316  0.586  0.724  0.771  1.00


#### 6.4 노상/노외별 점수 분포 인사이트

| 통계량 | 노외 주차장 (532개) | 노상 주차장 (209개) |
|--------|-----------------|-----------------|
| 평균 | 0.676 | 0.155 |
| 중앙값 | 0.724 | 0.115 |
| 표준편차 | 0.151 | 0.093 |
| 최솟값 | 0.316 | 0.029 |
| 최댓값 | 1.000 | 0.470 |

- 노외와 노상 간 평균 점수 차이 **0.52** → 두 그룹이 완전히 다른 분포
- 노상 최댓값(0.470)이 노외 25%값(0.586)에도 미치지 못함 →
  노상 주차장은 태양광 자가소비 적합성 측면에서 구조적으로 불리
- **원인**: 노상은 ① 단기 운영 비중 높아 운영시간 낮음 ② 조명·설비 상시소비 의무 없음
  ③ 소규모로 EV 충전기 설치 가능성 낮음 → 세 변수 모두 불리한 구조
- 노외 표준편차(0.151)가 노상(0.093)보다 크다 →
  노외 주차장 내에서도 EV충전 수준에 따라 편차가 존재

### 6.5 상위 20개 주차장 상세 비교

- 영등포동제2공영: EV충전 점수 압도적(내부 62개)
- 대부분 자체소비 1.0 + EV충전 차이로 순위 결정
- 신방화역, 용문동: 자체소비 점수로 상위권 진입

In [40]:
# ── 6.5 상위 20개 주차장 상세 비교 ──────────────────────────────────────
top20 = (df_result.sort_values('축3_자가소비적합성', ascending=False)
         .head(20)
         .reset_index(drop=True))
top20.index = top20.index + 1  # 순위 1부터 시작

print("=== 상위 20개 주차장 ===")
print(top20[['pklt_nm', '자치구', 'pklt_knd_nm',
             '운영시간', 'ev_충전기_설치수', 'ev_200m', 'ev_등록대수',
             '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성']].to_string())

=== 상위 20개 주차장 ===
           pklt_nm   자치구 pklt_knd_nm  운영시간  ev_충전기_설치수  ev_200m  ev_등록대수  자체소비_점수  EV충전_점수_norm  축3_자가소비적합성
1         영등포동제2공영  영등포구      노외 주차장  24.0          62       64     4323      1.0      1.000000    1.000000
2        잠실역 공영주차장   송파구      노외 주차장  24.0           0      182     7851      1.0      0.771839    0.931552
3       수서역북 공영주차장   강남구      노외 주차장  24.0           8       33    15468      1.0      0.725189    0.917557
4      탄천제2호 공영주차장   강남구      노외 주차장  24.0           0       57    15468      1.0      0.703662    0.911099
5       영희초교 공영주차장   강남구      노외 주차장  24.0           0       57    15468      1.0      0.703662    0.911099
6         영등포동제3공영  영등포구      노외 주차장  24.0          37       55     4323      1.0      0.685415    0.905624
7    역삼1문화센터 공영주차장   강남구      노외 주차장  24.0           0       44    15468      1.0      0.665387    0.899616
8   역삼문화공원제1호공영주차장   강남구      노외 주차장  24.0           0       44    15468      1.0      0.665387    0.899616
9     역삼문

In [43]:
# 상위 20개 수평 막대 + 최종점수 마커
top20_sorted = top20.sort_values('축3_자가소비적합성', ascending=True)

fig5 = go.Figure()

fig5.add_trace(go.Bar(
    name='자체소비 점수',
    y=top20_sorted['pklt_nm'],
    x=top20_sorted['자체소비_점수'],
    orientation='h',
    marker_color='#1a5fa8',
    opacity=0.8
))

fig5.add_trace(go.Bar(
    name='EV충전 점수',
    y=top20_sorted['pklt_nm'],
    x=top20_sorted['EV충전_점수_norm'],
    orientation='h',
    marker_color='#FF9800',
    opacity=0.8
))

fig5.add_trace(go.Scatter(
    name='축3 최종점수',
    y=top20_sorted['pklt_nm'],
    x=top20_sorted['축3_자가소비적합성'],
    mode='markers',
    marker=dict(size=10, color='#E74C3C', symbol='diamond'),
))

fig5.update_layout(
    title='상위 20개 주차장 축3 변수별 상세 비교',
    xaxis_title='점수',
    barmode='overlay',
    height=650,
    legend=dict(x=0.7, y=0.1),
    yaxis=dict(autorange='reversed')
)
fig5.show()
fig5.write_html(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\output\figures\상위20_축3_상세비교.html'
)

#### 6.5 상위 20개 주차장 인사이트

**공통 특징**
- 전체 20개 모두 **노외 주차장 + 24시간 운영** → 자체소비 점수 1.0으로 수렴
- 상위 20개 내 순위는 **EV충전 점수가 전적으로 결정**

**주목할 주차장**

| 순위 | 주차장명 | 특징 |
|------|---------|------|
| 1 | 영등포동제2공영 | ev_충전기_설치수 62개(압도적 1위) + ev_200m 64개 → EV충전 점수 1.0 |
| 2 | 잠실역 공영주차장 | ev_충전기_설치수 0개이나 ev_200m 182개(2위) + ev_등록대수 7851 |
| 3 | 수서역북 공영주차장 | ev_충전기_설치수 8개 + ev_등록대수 15468(강남구 최상위) 복합 효과 |
| 4~5 | 탄천제2호·영희초교 | ev_충전기_설치수 0개이나 강남구 EV 등록대수(15468) 효과로 상위권 |
| 14 | 천호역 공영주차장 | ev_충전기_설치수 21개 + ev_200m 72개 → EV 인프라 양면 강점 |

**누적 막대 해석**
- EV충전 점수(주황)가 길수록 실제 EV 인프라가 풍부한 입지
- 자체소비(파랑)는 상위 20개 전체 동일 수준 → 변별력 없음
- 영등포동제2공영만 유일하게 EV충전 점수가 자체소비 점수를 압도 →
  내부 충전기 62개라는 독보적 인프라 보유
- 강남구 주차장 다수(7~17위)가 ev_충전기_설치수=0임에도 상위권 →
  ev_등록대수(15468) 효과가 강남구 전체를 끌어올림

---
### 6.6.1 보완 분석 — ev_충전기_설치수 프록시 정확도 검토1

#### 발단

분석 완료 후 서울시 보도자료([news.seoul.go.kr/env/archives/525227](https://news.seoul.go.kr/env/archives/525227))를 검토하는 과정에서
신천유수지 공영주차장(송파구)에 실제 EV 충전기가 설치되어 있음을 확인하였다.
그러나 해당 주차장의 ev_충전기_설치수가 **0개**로 집계되어 있어
50m 거리 기반 프록시의 정확도를 재검토하였다.

---

#### 검토 방법

보도자료 및 환경부 API 데이터(서울_EV충전소.csv) 교차 확인을 통해
실제 EV 충전기가 설치된 것으로 확인되나 50m 집계에서 누락된 주차장을
수동으로 탐색하였다.

- **탐색 범위**: 서울시 보도자료 기반 충전기 설치 공영주차장
- **확인 방법**: 환경부 API 내 충전소명 검색 → 주차장 좌표와 거리 측정
- **한계**: 전체 741개 주차장 전수 검토가 아닌 보도자료 기반 부분 탐색으로
  누락 주차장이 더 존재할 가능성 있음


In [6]:
# EV충전소 데이터에서 6개 주차장 좌표 확인
target_ev_names = [
    '동작갯마을 공영주차장',
    '홍은2동제3공영주차장',
    '서울은평 수색동 공영주차장',
    '문정근린공원 공영주차장',
    '수서역 공영주차장 복합충전소',
    '신천유수지공영주차장'
]

target_ev = df_ev[df_ev['statNm'].isin(target_ev_names)][['statNm', 'lat', 'lng']].drop_duplicates()
print("=== EV충전소 데이터 내 좌표 ===")
print(target_ev.to_string())

=== EV충전소 데이터 내 좌표 ===
                statNm        lat         lng
908        동작갯마을 공영주차장  37.495826  126.982529
3084       홍은2동제3공영주차장  37.586712  126.934149
16267       신천유수지공영주차장  37.521743  127.102254
22228   서울은평 수색동 공영주차장  37.580428  126.898810
24740     문정근린공원 공영주차장  37.485960  127.123629
63316  수서역 공영주차장 복합충전소  37.487567  127.100724
63494     문정근린공원 공영주차장  37.485949  127.123894
72990       신천유수지공영주차장  37.521700  127.102300


In [85]:
# 우리 df에서 대응되는 주차장 찾기 (이름 유사한 것)
keywords = ['신천유수지', '수서역', '문정근린', '수색동', '홍은2동', '동작갯마을']

print("=== df 내 대응 주차장 ===")
for kw in keywords:
    matches = df[df['pklt_nm'].str.contains(kw, na=False)][['pklt_nm', 'lat', 'lot']]
    if len(matches) > 0:
        print(matches.to_string())
    else:
        print(f"{kw}: 매칭 없음")

=== df 내 대응 주차장 ===
         pklt_nm        lat         lot
451  신천유수지 공영주차장  37.522315  127.102903
        pklt_nm        lat        lot
414  수서역북 공영주차장  37.488013  127.09995
    pklt_nm       lat        lot
282  문정근린공원  37.48434  127.12246
      pklt_nm        lat         lot
413  수색동공영주차장  37.585187  126.894858
          pklt_nm       lat        lot
707  홍은2동 제1공영주차장  37.59296  126.93733
708  홍은2동 제3공영주차장  37.59296  126.93733
         pklt_nm        lat         lot
221        동작갯마을  37.502891  126.977529
222  동작갯마을 공영주차장  37.502891  126.977529


In [86]:
# 50m vs 100m 집계 비교
print("=== 50m vs 100m 집계 비교 ===\n")

for kw in keywords:
    park = df[df['pklt_nm'].str.contains(kw, na=False)]
    if len(park) == 0:
        print(f"{kw}: df 매칭 없음")
        continue
    
    for _, row in park.iterrows():
        dists = [haversine(row['lat'], row['lot'], elat, elng)
                 for elat, elng in zip(ev_lat, ev_lng)]
        cnt_50  = sum(d <= 50  for d in dists)
        cnt_100 = sum(d <= 100 for d in dists)
        print(f"{row['pklt_nm']}")
        print(f"  50m:  {cnt_50}개")
        print(f"  100m: {cnt_100}개")
        print()

=== 50m vs 100m 집계 비교 ===

신천유수지 공영주차장
  50m:  0개
  100m: 8개

수서역북 공영주차장
  50m:  8개
  100m: 22개

문정근린공원
  50m:  5개
  100m: 40개

수색동공영주차장
  50m:  0개
  100m: 31개

홍은2동 제1공영주차장
  50m:  0개
  100m: 0개

홍은2동 제3공영주차장
  50m:  0개
  100m: 0개

동작갯마을
  50m:  0개
  100m: 0개

동작갯마을 공영주차장
  50m:  0개
  100m: 0개



#### 탐색 결과 — 50m vs 100m 집계 비교

환경부 API 내 대응 충전소명:

| 우리 데이터 주차장명 | 환경부 API 충전소명 | 50m | 100m |
|------------------|-----------------|-----|------|
| 신천유수지 공영주차장 | 신천유수지공영주차장 | 0개 | 8개 |
| 수서역북 공영주차장 | 수서역 공영주차장 복합충전소 | 8개 | 22개 |
| 문정근린공원 공영주차장 | 문정근린공원 공영주차장 | 5개 | 40개 |
| 수색동공영주차장 | 서울은평 수색동 공영주차장 | 0개 | 31개 |
| 홍은2동 제3공영주차장 | 홍은2동제3공영주차장 | 0개 | 0개 |
| 동작갯마을 공영주차장 | 동작갯마을 공영주차장 | 0개 | 0개 |

---

#### 결과 해석

**반경 확대(100m)로 해결 가능한 경우:**
- 신천유수지, 수색동 → 50m 좌표 오차로 누락, 100m에서 포착

**반경 확대로도 해결 불가한 경우:**
- 홍은2동 제3공영, 동작갯마을 → 100m에서도 0개
- 환경부 API 미등록이거나 충전소 좌표가 주차장과 전혀 다른 위치에 등록된 것으로 추정

**반경 확대 시 과대추정 심화:**
- 문정근린공원: 5개 → 40개 (8배 폭증)
- 수서역북: 8개 → 22개 (2.75배 증가)
- 주차장 경계 밖 인근 건물·도로변 충전소까지 혼입되는 구조

---

#### 결론 — 50m 유지

반경을 100m로 확대할 경우:
- 일부 누락(신천유수지·수색동)은 해결되나
- 홍은2동·동작갯마을은 여전히 해결 불가
- 문정근린공원 등에서 과대추정이 심화되어 전체 정확도가 오히려 저하됨

**반경 조정으로는 해결이 어려운 구조적 한계**로 판단하여 50m 기준을 유지한다.
근본 원인은 반경 크기가 아닌 **환경부 API 충전소 좌표와 주차장 좌표 간
등록 기준 불일치**에 있다.

> **한계 명시**: 본 분석의 ev_충전기_설치수는 50m 거리 기반 프록시로,
> 실제 설치된 충전기가 있음에도 좌표 불일치로 인해 0개로 집계되는
> 과소추정 사례가 존재한다. 충전소 좌표 데이터의 정확도 향상 또는
> 주차장별 실측 데이터 공개 시 보완 가능하다.

### 6.6.2 보완 분석 — ev_충전기_설치수 프록시 정확도 검토2

**50m 0개 비율(84.4%)의 현실적 해석**

In [ ]:
# ev_충전기_설치수 > 0 주차장
ev_parks = (df_result[df_result['ev_충전기_설치수'] > 0]
            .sort_values('ev_충전기_설치수', ascending=False)
            .reset_index(drop=True))
ev_parks.index = ev_parks.index + 1  # 순위 1부터

print(f"ev_충전기_설치수 > 0 주차장: {len(ev_parks)}개 / 741개 ({len(ev_parks)/741*100:.1f}%)")
print()
print(ev_parks[['pklt_nm', '자치구', 'pklt_knd_nm',
                'ev_충전기_설치수', 'ev_200m', 'ev_등록대수',
                '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성']].to_string())

#### 외부 통계와의 비교 — 과소추정 확인

서울시 보도자료(2021)에 따르면 50면 이상 자치구 소유·관리 공영주차장
328개소에 349기가 설치되어 주차장당 평균 약 1기 수준이다.

반면 본 분석의 50m 프록시 기준으로는 741개소 중 116개소(15.7%),
평균 0.87개가 집계되어 위 통계와 유사한 수준으로 나타났다.

| 구분 | 기준 | EV 충전기 현황 |
|------|------|--------------|
| 서울시 보도자료 (2021) | 50면 이상 공영주차장 328개소 | 349기 (주차장당 평균 1.06기) |
| 본 분석 50m 프록시 | 분석 대상 741개소 | 116개소 (15.7%), 평균 0.87개 |


**해석:**
- 두 수치가 유사한 수준 → 50m 프록시가 실제 보급 현황을
  대체로 반영하고 있음을 간접적으로 확인
- 다만 50m 프록시는 좌표 오차로 인해 실제 내부 충전기를
  누락하거나 외부 충전기를 혼입하는 양방향 오류가 존재
- ev_충전기_설치수는 정확한 내부 충전기 수가 아닌
  **보수적 프록시 추정값**으로 해석해야 한다

출처: [text](https://news.seoul.go.kr/env/archives/515890)

| 구분 | 전체 공영주차장 | EV 충전기 설치 현황 | 출처 |
|------|--------------|-----------------|------|
| 2021년 12월 | 328개소 (50면 이상 자치구 공영) | 349기 설치 | 서울시 보도자료 [text](https://news.seoul.go.kr/env/archives/515890) |
| 2021년 계획 | 328개소 (50면 이상) | 208개소 추가, 592기 추가 계획 | 이데일리 [text](https://m.edaily.co.kr/News/Read?newsId=02233686629274256&mediaCodeNo=257) |
| 2025년 5월 | - | 급속 645기 (공영 일부) | 서울시 보도자료 [text](https://news.seoul.go.kr/env/archives/525227)|

→ 공영주차장 EV 충전기 보급이 지속 확대 중
→ 본 분석 50m 프록시(116개소, 15.7%)는 보수적 하한 추정값

---
## 7. 분석 요약 및 결론

### 7.1 분석 목적 및 범위 요약

본 분석은 서울시 공영주차장 741개소를 대상으로
태양광 캐노피 설치 시 **자가소비 적합성**을 평가하는 축3 점수를 산출하였다.

- **분석 범위**: 태양광 전력의 직접 소비 가능한 두 가지 경로로 한정
  - 주차장 자체 소비 (조명·CCTV·출입기 등 설비 전력)
  - EV 충전기 직접 공급 (태양광 → 인버터 → 배전반 → EV 충전기)

---

### 7.2 최종 가중치 확정

| 단계 | 변수 | 가중치 | 산정 방법 |
|------|------|--------|---------|
| 자체소비 점수 | 운영시간 | 0.6 | 상관관계 + 민감도 분석 |
| 자체소비 점수 | 노외여부 | 0.4 | 상관관계 + 민감도 분석 |
| EV충전 점수 | ev_충전기_설치수 (50m) | 0.4 | 상관관계 + 민감도 분석 |
| EV충전 점수 | ev_200m | 0.3 | 상관관계 + 민감도 분석 |
| EV충전 점수 | ev_등록대수 | 0.3 | 상관관계 + 민감도 분석 |
| **축3 최종** | 자체소비 점수 | **0.7** | **CRITIC** |
| **축3 최종** | EV충전 점수 | **0.3** | **CRITIC** |

> **방법론 선택 근거**
> - 자체소비·EV충전 그룹 내부: 이진변수(노외여부) 및 집계단위 불일치(ev_등록대수)로
>   CRITIC 적용 시 표준편차 왜곡 확인 → 상관관계 + 민감도 분석으로 대체
> - 하위축 간: 두 점수 모두 연속형 + 주차장별 개별값으로 CRITIC 적합
>   → CRITIC 결과(자체소비 0.696 / EV충전 0.304) 반올림 채택

---

### 7.3 점수 분포 요약

| 통계량 | 자체소비 점수 | EV충전 점수(정규화) | 축3 최종 점수 |
|--------|------------|-----------------|------------|
| 평균 | 0.670 | 0.199 | 0.575 |
| 표준편차 | 0.382 | 0.171 | 0.307 |
| 중앙값 | 1.000 | 0.147 | 0.801 |
| 최댓값 | 1.000 | 1.000 | 1.000 |

---

### 7.4 핵심 발견

**1. 자체소비 점수가 상위권 진입의 필요조건**
- 24시간 노외 주차장(382개, 51.6%)이 자체소비 점수 1.0으로 수렴
- 상위 20개 주차장 전원 노외 + 24시간 운영
- 노외 평균(0.676) vs 노상 평균(0.155) → 0.52 차이로 두 그룹이 완전히 분리

**2. EV충전 점수가 상위권 내 순위를 결정하는 변별 요소**
- 자체소비 점수가 수렴한 상위권에서 최종 순위는 EV충전 점수로 결정
- 영등포동제2공영: ev_충전기_설치수 62개로 EV충전 점수 1.0 → 종합 1위
- 강남구: ev_충전기_설치수=0이어도 ev_등록대수(15,468대)로 상위권 다수 진입

**3. 자치구 평균과 개별 입지 순위의 역전 현상**
- 영등포구: 자치구 평균 하위(0.327)이나 영등포동제2·3공영은 전체 1·6위
- 은평·성북: 주차장 수가 적어 24시간 노외 위주로 구성 → 평균 상위권
- **자치구 단위 정책보다 개별 입지 선정이 더 중요함을 시사**

---

### 7.5 분석 한계

| 한계 | 내용 | 향후 보완 방향 |
|------|------|-------------|
| ev_충전기_설치수 프록시 한계 | 50m 거리 기반 집계, 실측값과 절대 평균 차이 6.64개 | 공개 데이터 확보 시 실측값으로 대체 |
| ev_등록대수 집계단위 불일치 | 자치구별 단일값을 주차장에 반복 매핑 | 동 단위 또는 주차장 반경 기반 등록대수로 정밀화 |
| 구조안전성 미반영 | 캐노피 하중 추가 가능 여부 미검토 | 최종 설치 전 구조안전진단 별도 필요 |
| 실제 전력소비량 미반영 | 주차장별 실제 전력사용량 데이터 미공개 | 한전 스마트미터 데이터 연계 시 정밀화 가능 |
| EV충전 점수 분포 편중 | ev_충전기_설치수 0개 주차장 84.4%로 EV충전 점수가 하단에 집중 | 서울시 공영주차장 EV 충전기 보급 확대 시 점수 분포 정상화 가능 |

---

### 7.6 다음 단계

본 축3 결과는 축1(태양광 발전 적합성), 축2(시설 전환 가능성),
축4(공공편익 접근성) 점수와 통합되어 **최종 종합점수** 산출에 활용된다.


In [44]:
# 축3 최종 저장 확인
print(f"저장 파일: parking_axis3_scored.csv")
print(f"행 수: {len(df_result)}개")
print(f"컬럼: {df_result.columns.tolist()}")
print(f"\n축3 점수 최종 분포:")
print(df_result['축3_자가소비적합성'].describe().round(3))

저장 파일: parking_axis3_scored.csv
행 수: 741개
컬럼: ['pklt_nm', 'pklt_knd_nm', 'lat', 'lot', '자치구', '운영시간', '노외여부', 'ev_충전기_설치수', 'ev_200m', 'ev_등록대수', '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성', '주차장유형']

축3 점수 최종 분포:
count    741.000
mean       0.529
std        0.272
min        0.029
25%        0.309
50%        0.703
75%        0.744
max        1.000
Name: 축3_자가소비적합성, dtype: float64


---

### 추가)특정 주차장에 대한 정보 탐색

In [1]:
import pandas as pd

df_result = pd.read_csv(
    r'C:\Users\seonu\Documents\axis3-self-consumption\canopy\data\processed\parking_axis3_scored.csv',
    encoding='utf-8-sig'
)

# 4분면 분류 (저장 파일에 분면 컬럼 없으면 재산출)
mid_x = df_result['자체소비_점수'].mean()
mid_y = df_result['EV충전_점수_norm'].mean()

df_result['분면'] = df_result.apply(lambda r:
    '최우선 (자소↑EV↑)'  if r['자체소비_점수'] >= mid_x and r['EV충전_점수_norm'] >= mid_y else
    'EV강점 (자소↓EV↑)'  if r['자체소비_점수'] <  mid_x and r['EV충전_점수_norm'] >= mid_y else
    '자소강점 (자소↑EV↓)' if r['자체소비_점수'] >= mid_x and r['EV충전_점수_norm'] <  mid_y else
    '낮은적합 (자소↓EV↓)', axis=1
)

print(f"로드 완료: {len(df_result)}개")
print(df_result.columns.tolist())

로드 완료: 741개
['pklt_nm', 'pklt_knd_nm', 'lat', 'lot', '자치구', '운영시간', '노외여부', 'ev_충전기_설치수', 'ev_200m', 'ev_등록대수', '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성', '분면']


In [2]:
def search_parking(name):
    result = df_result[df_result['pklt_nm'].str.contains(name, na=False)]
    
    if len(result) == 0:
        print(f"'{name}' 검색 결과 없음")
        return
    
    cols = ['pklt_nm', '자치구', 'pklt_knd_nm',
            '운영시간', '노외여부',
            'ev_충전기_설치수', 'ev_200m', 'ev_등록대수',
            '자체소비_점수', 'EV충전_점수_norm', '축3_자가소비적합성', '분면']
    
    for _, row in result.iterrows():
        print("=" * 50)
        print(f"주차장명    : {row['pklt_nm']}")
        print(f"자치구      : {row['자치구']}  |  유형: {row['pklt_knd_nm']}")
        print(f"운영시간    : {row['운영시간']:.2f}h  |  노외여부: {'노외' if row['노외여부']==1 else '노상'}")
        print("-" * 50)
        print(f"[변수]")
        print(f"  ev_충전기_설치수 (50m) : {int(row['ev_충전기_설치수'])}개")
        print(f"  ev_200m               : {int(row['ev_200m'])}개")
        print(f"  ev_등록대수           : {int(row['ev_등록대수'])}대")
        print("-" * 50)
        print(f"[점수]")
        print(f"  자체소비_점수         : {row['자체소비_점수']:.3f}")
        print(f"  EV충전_점수_norm      : {row['EV충전_점수_norm']:.3f}")
        print(f"  축3_자가소비적합성    : {row['축3_자가소비적합성']:.3f}")
        print(f"  분면                  : {row['분면']}")
        print("=" * 50)

# 사용 예시
search_parking("영등포동제2")
search_parking("잠실역")

주차장명    : 영등포동제2공영
자치구      : 영등포구  |  유형: 노외 주차장
운영시간    : 24.00h  |  노외여부: 노외
--------------------------------------------------
[변수]
  ev_충전기_설치수 (50m) : 62개
  ev_200m               : 64개
  ev_등록대수           : 4323대
--------------------------------------------------
[점수]
  자체소비_점수         : 1.000
  EV충전_점수_norm      : 1.000
  축3_자가소비적합성    : 1.000
  분면                  : 최우선 (자소↑EV↑)
주차장명    : 잠실역 공영주차장
자치구      : 송파구  |  유형: 노외 주차장
운영시간    : 24.00h  |  노외여부: 노외
--------------------------------------------------
[변수]
  ev_충전기_설치수 (50m) : 0개
  ev_200m               : 182개
  ev_등록대수           : 7851대
--------------------------------------------------
[점수]
  자체소비_점수         : 1.000
  EV충전_점수_norm      : 0.772
  축3_자가소비적합성    : 0.932
  분면                  : 최우선 (자소↑EV↑)
